Dùng docling để đọc file pdf   

1   Dùng docling để tìm ra các heading và vị trí của: heading, đoạn code, đoạn text, listitem, hình ảnh - kèm caption, công thức toán, bảng biểu - kèm caption.

    1.1 thư viện
    1.2 các element được chọn - nhận diện heading, code, công thức toán, hình ảnh, bảng biểu
    1.3 các công cụ của docling dược dùng
    1.4 hàm xác định các heading tìm được bằng docling và đặc điểm của chúng
    1.5 hàm xác định thêm cho các heading vừa tìm được các đặc điểm mới giờ đây ta được các heading toàn file pdf nhưng bị thiếu 1 số heading do docling nhận nhầm
    1.6 cell mới
    1.7 cell mới
    1.8 Tìm các heading trong file pdf mà bị docling nhận diện nhầm thành text
    1.9 gộp các heading bị nhận diện nhầm ở 6 vào 5 theo đúng vị trí mà nó nên có và tạo ra được bảng các heading trong toàn file pdf

=> sau 9 bước này chúng ta tìm được tất cả các

    heading trên toàn file pdf và vị trí của đoạn
    code, hình ảnh, công thức toán, bảng biểu, listitem, đoạn text




2   Thử kiểm tra xem file pdf bên trên có bookmark không, nếu không có thì tự tìm kiếm table of content/mục lục --- làm cái này bởi vì phải có cấu trúc cây thì mới dễ dàng phân vùng và tìm kiếm.

    2.1 thư viện
    2.2 nếu có bookmark tìm và kiểm tra bookmark và xuất mục lục thành dạng cây
    2.3 nếu không có thì dùng hàm này để tìm các heading trong mục lục/table of content và kèm theo các đặc điểm của chúng - để lúc sau có thể chia nhóm và chia theo cấu trúc cây
    2.4 trong heading thì có những heading có chỉ số và có những cái không có chỉ số thì đoạn code này sẽ gồm các hàm định nghĩa chỉ số và tìm những heading có chỉ số và không có chỉ số sau đó chia phân nhóm các heading đó  
    2.5 sắp xếp các nhóm đã tìm được ở bước trên và tạo cấu trúc cây và bậc cho chúng  
    2.6 soát lại 1 lượt các heading trong toàn file pdf đã tìm được được lưu trong struct_merged xem những heading nào bị trùng với các font monospace
    2.7 cell mới
    2.8 hàm tìm các heading bậc có dạng giống với heading bậc nhỏ nhất trong heading_muc_luc_tree
    2.9 hàm tìm các heading bậc có dạng giống với heading bậc nhỏ nhất trong heading_muc_luc_tree, gán cho các heading tìm được có bậc nhỏ hơn bậc của heading bậc nhỏ nhất đó 1 bậc
    2.10 không dùng 2 hàm trong 2.9 và 2.10 nữa thay vào đó là dùng hàm này để
        **gán đúng cấp bậc (level) cho tất cả các heading được tìm thấy trong file PDF**
    2.11 hàm này để xác định các heading còn sót level ở hàm trước trong struct_merged --- nó kiểu những "heading tự do"
    2.12 hàm tìm font_name
    2.13 xác định bậc cho các heading none còn lại trong struct_merged

# **cell luồng xử lý cho heading ra được --- cái này chứa luồng chạy tất cả các hàm bên trên**

=> đầu ra là

    struct_merged, real_heading_muc_luc_level, heading_muc_luc_tree, element_coords = tất cả heading trong file pdf được gọn gàng, các heading trong mục lục và level , cấu trúc tree của mục lục, vị trí đoạn code + text + hình ảnh + list + công thức toán. --- hết hàm 2.10

    bậc của các heading trong struct_merged -- tức là tất cả các heading của file pdf, mỗi heading đều chứa đặc điểm font_name, font_size --- hết hàm 2.13



1   Dùng docling để tìm ra các heading và vị trí của: heading, đoạn code, đoạn text, listitem, hình ảnh - kèm caption, công thức toán, bảng biểu - kèm caption.

1.1 thư viện     

In [ ]:
!pip install docling docling-hierarchical-pdf
!pip install pdfplumber pymupdf -q
!pip uninstall -y pillow
!pip install -U pillow --no-cache-dir

In [ ]:
!pip install pypdf -q
!pip install pdfplumber -q
!pip install pymupdf

1.2 các element được chọn

In [ ]:
import torch
from docling_core.types.doc import DocItemLabel

LABEL_MAP = {
    DocItemLabel.TITLE:               "TITLE",
    DocItemLabel.SECTION_HEADER:      "SECTION_HEADER",
    DocItemLabel.TEXT:                "TEXT",
    DocItemLabel.LIST_ITEM:           "LIST_ITEM",
    DocItemLabel.TABLE:               "TABLE",
    DocItemLabel.PICTURE:             "PICTURE",
    DocItemLabel.FORMULA:             "FORMULA",
    DocItemLabel.CAPTION:             "CAPTION",
    DocItemLabel.FOOTNOTE:            "FOOTNOTE",
    DocItemLabel.PAGE_HEADER:         "PAGE_HEADER",
    DocItemLabel.PAGE_FOOTER:         "PAGE_FOOTER",
    DocItemLabel.CODE:                "CODE",
    DocItemLabel.KEY_VALUE_REGION:    "KEY_VALUE",
    DocItemLabel.CHECKBOX_SELECTED:   "CHECKBOX_CHECKED",
    DocItemLabel.CHECKBOX_UNSELECTED: "CHECKBOX_UNCHECKED",
}

COLOR_MAP = {
    "TITLE":              (1.0,  0.42, 0.42),
    "SECTION_HEADER":     (1.0,  0.62, 0.26),
    "TEXT":               (0.33, 0.63, 1.0),
    "LIST_ITEM":          (0.37, 0.15, 0.80),
    "TABLE":              (0.0,  0.82, 0.83),
    "PICTURE":            (0.11, 0.82, 0.63),
    "FORMULA":            (0.95, 0.41, 0.88),
    "CAPTION":            (0.99, 0.79, 0.34),
    "FOOTNOTE":           (0.78, 0.84, 0.90),
    "PAGE_HEADER":        (0.51, 0.58, 0.65),
    "PAGE_FOOTER":        (0.34, 0.40, 0.45),
    "CODE":               (0.93, 0.35, 0.14),
    "KEY_VALUE":          (0.0,  0.58, 0.20),
    "CHECKBOX_CHECKED":   (0.64, 0.80, 0.22),
    "CHECKBOX_UNCHECKED": (0.82, 0.85, 0.88),
}

print("Sẵn sàng!")

1.3 các công cụ của docling được dùng

In [ ]:
# from google.colab import files
# from docling.document_converter import DocumentConverter, PdfFormatOption
# from docling.datamodel.base_models import InputFormat
# from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode  # ← thêm TableFormerMode vào đây
# from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

# uploaded = files.upload()
# pdf_path = list(uploaded.keys())[0]
# print(f"File: {pdf_path}")

# pipeline_options = PdfPipelineOptions()
# pipeline_options.do_ocr = False
# pipeline_options.do_table_structure = True
# pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE  # ← thêm dòng này
# pipeline_options.generate_picture_images = True
# pipeline_options.images_scale = 2.0   # ← render trang ở độ phân giải cao hơn trước khi feed vào model


# converter = DocumentConverter(
#     format_options={
#         InputFormat.PDF: PdfFormatOption(
#             pipeline_options=pipeline_options,
#             backend=PyPdfiumDocumentBackend
#         )
#     }
# )

# result = converter.convert(pdf_path)
# doc = result.document
# print(" Xong!")

1.4 hàm xác định các heading tìm được bằng docling và các đặc điểm của chúng

In [ ]:
import fitz
from docling_core.types.doc import SectionHeaderItem
from collections import Counter


# ── Các hàm xử lý char-level (xem giải thích chi tiết ở heading_extract.py) ──

def extract_chars_in_bbox(page, rect):
    d = page.get_text("rawdict", clip=rect)
    chars = []
    for block in d.get("blocks", []):
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                size = span["size"]
                for ch in span.get("chars", []):
                    c = ch["c"]
                    if c.strip() == "" and c != " ":
                        continue
                    x0, y0, x1, y1 = ch["bbox"]
                    origin_x, origin_y = ch["origin"]
                    chars.append({
                        "char": c,
                        "x0": x0, "y0": y0, "x1": x1, "y1": y1,
                        "origin_x": origin_x,
                        "origin_y": origin_y,
                        "size": size,
                    })
    return chars


def classify_track(chars, baseline_tolerance=1.5):
    if not chars:
        return chars
    rounded = [round(c["origin_y"]) for c in chars]
    main_baseline = Counter(rounded).most_common(1)[0][0]
    for c in chars:
        dy = c["origin_y"] - main_baseline
        if abs(dy) <= baseline_tolerance:
            c["track"] = "main"
        elif dy < 0:
            c["track"] = "sup"
        else:
            c["track"] = "sub"
    return chars


def chars_to_text(chars, mark_sup_sub=False, space_gap_ratio=0.25):
    if not chars:
        return ""
    chars_sorted = sorted(chars, key=lambda c: c["x0"])

    out = []
    prev = None
    for c in chars_sorted:
        if prev is not None:
            gap = c["x0"] - prev["x1"]
            # Nếu khoảng trống giữa 2 ký tự đủ lớn so với size font → coi là có space
            threshold = prev["size"] * space_gap_ratio
            if gap > threshold and c["char"] != " " and prev["char"] != " ":
                out.append(" ")
        out.append(c["char"])
        prev = c
    return "".join(out)


def _wrap(s, track):
    if track == "sup":
        return f"^{{{s}}}"
    if track == "sub":
        return f"_{{{s}}}"
    return s


def get_heading_text_from_bbox(page, rect, mark_sup_sub=False):
    chars = extract_chars_in_bbox(page, rect)
    chars = classify_track(chars)
    text = chars_to_text(chars, mark_sup_sub=mark_sup_sub)
    dominant_size = None
    if chars:
        size_count = Counter()
        for c in chars:
            size_count[round(c["size"], 1)] += 1
        dominant_size = size_count.most_common(1)[0][0]
    return text, dominant_size, chars


# ── Pipeline chính: trích heading từ Docling, đọc lại text qua bbox ─────────

def build_struct_raw(doc, pdf_path, mark_sup_sub=False):
    pdf = fitz.open(pdf_path)
    rows = []

    for item, level in doc.iterate_items():
        if not isinstance(item, SectionHeaderItem):
            continue
        if not item.prov:
            continue

        prov = item.prov[0]
        page_no = prov.page_no
        bbox = prov.bbox
        page = pdf[page_no - 1]
        page_height = page.rect.height

        # Docling bbox: gốc dưới-trái (PDF coords) -> đổi sang PyMuPDF (gốc trên-trái)
        rect = fitz.Rect(
            bbox.l,
            page_height - bbox.t,
            bbox.r,
            page_height - bbox.b
        )

        text, dominant_size, chars = get_heading_text_from_bbox(
            page, rect, mark_sup_sub=mark_sup_sub
        )

        rows.append({
            "level"        : item.level,
            "text"         : text,
            "page"         : page_no,
            "font_size"    : dominant_size,
            "bbox_l"       : round(bbox.l, 1),
            "bbox_t"       : round(bbox.t, 1),
            "bbox_r"       : round(bbox.r, 1),
            "bbox_b"       : round(bbox.b, 1),
            "chars_detail" : chars,   # debug: xem từng ký tự + track nếu cần
        })

    # ── In ra bảng + lưu struct_raw ──────────────────────────────────────────
    print(f"{'H':<4} {'Trang':<6} {'Size':<6} {'Left_Top':<14} {'Left_Bot':<14} {'Text'}")
    print("-" * 90)

    struct_raw = []
    for r in rows:
        indent = "  " * (r["level"] - 1)
        left_top = f"({r['bbox_l']}, {r['bbox_t']})"
        left_bot = f"({r['bbox_l']}, {r['bbox_b']})"
        line = (f"H{r['level']:<3} {r['page']:<6} {str(r['font_size']):<6} "
                f"{left_top:<14} {left_bot:<14} {indent}{r['text']}")
        print(line)
        struct_raw.append({
            "level"        : r["level"],
            "page"         : r["page"],
            "font_size"    : r["font_size"],
            "left_top"     : (r["bbox_l"], r["bbox_t"]),
            "left_bot"     : (r["bbox_l"], r["bbox_b"]),
            "text"         : r["text"],
            "chars_detail" : r["chars_detail"],
        })

    print(f"\nĐã lưu {len(struct_raw)} heading vào struct_raw")
    return struct_raw, rows


# # ── Cách dùng ─────────────────────────────────────────────────────────────────
# struct_raw, rows = build_struct_raw(doc, pdf_path, mark_sup_sub=False)


1.5 hàm xác định thêm cho các heading vừa tìm được các đặc điểm mới giờ đây ta được các heading toàn file pdf nhưng bị thiếu 1 số heading do docling nhận nhầm

In [ ]:
import fitz
import re
from collections import defaultdict
from docling_core.types.doc import SectionHeaderItem, TextItem, DocItemLabel

# ── helpers ──────────────────────────────────────────────────────────────────
def is_bold(span: dict) -> bool:
    if span["flags"] & 16:
        return True
    fname = span["font"].lower()
    return any(kw in fname for kw in ("bold", "black", "heavy", "demi", "semibold"))

def base_font(font_name: str) -> str:
    return re.sub(r"[-,]?(bold|italic|regular|roman|light|medium|black|heavy|demi|semibold|oblique).*",
                  "", font_name, flags=re.IGNORECASE).strip()

def dominant_span_info(page, bbox, page_height):
    rect = fitz.Rect(bbox.l, page_height - bbox.t, bbox.r, page_height - bbox.b)
    best = None
    best_len = -1
    for b in page.get_text("dict", clip=rect)["blocks"]:
        for line in b.get("lines", []):
            for span in line.get("spans", []):
                t = span["text"].strip()
                if not t:
                    continue
                if len(t) > best_len:
                    best_len = len(t)
                    best = span
    if best is None:
        return None
    return {
        "font"     : best["font"],
        "base_font": base_font(best["font"]),
        "size"     : round(best["size"], 1),
        "bold"     : is_bold(best),
    }

# ── 1. HÀM CHÍNH: Thu thập title / section_header đậm ────────────────────────
def get_bold_headings(doc, pdf_path):
    pdf = fitz.open(pdf_path)
    heading_signatures = set()
    heading_rows = []

    for item, level in doc.iterate_items():
        if item.label not in (DocItemLabel.TITLE, DocItemLabel.SECTION_HEADER):
            continue
        if not item.prov:
            continue
        prov  = item.prov[0]
        bbox  = prov.bbox
        page  = pdf[prov.page_no - 1]
        info  = dominant_span_info(page, bbox, page.rect.height)
        if info is None or not info["bold"]:
            continue

        sig = (info["base_font"], info["size"])
        heading_signatures.add(sig)
        heading_rows.append({
            "label"    : item.label.value,
            "page"     : prov.page_no,
            "font"     : info["font"],
            "base_font": info["base_font"],
            "size"     : info["size"],
            "bbox_l"   : round(bbox.l, 1),
            "bbox_t"   : round(bbox.t, 1),
            "bbox_b"   : round(bbox.b, 1),
            "text"     : getattr(item, "text", "")[:80],
        })

    print(f"=== HEADING ĐẬM ({len(heading_rows)} dòng) ===")

    heading_raw_elements = []

    for r in heading_rows:
        heading_raw_elements.append({
            "label"    : r["label"],
            "page"     : r["page"],
            "size"     : r["size"],
            "left_top" : (r["bbox_l"], r["bbox_t"]),
            "left_bot" : (r["bbox_l"], r["bbox_b"]),
            "font"     : r["font"],
            "base_font": r["base_font"],
            "text"     : r["text"],
        })

    print(f"Signatures cần khớp: {heading_signatures}")
    print(f"Đã lưu {len(heading_raw_elements)} heading vào heading_raw_elements")

    return heading_raw_elements, heading_signatures

1.6 cell mới

In [ ]:
import fitz
from docling_core.types.doc import DocItemLabel

def bold_not_preceded_by_normal(page, bbox, page_height) -> bool:
    rect = fitz.Rect(bbox.l, page_height - bbox.t, bbox.r, page_height - bbox.b)
    blocks = page.get_text("dict", clip=rect)["blocks"]
    for b in blocks:
        for line in b.get("lines", []):
            spans = [s for s in line.get("spans", []) if s["text"].strip()]
            found_bold = False
            for span in spans:
                if is_bold(span):
                    found_bold = True
                else:
                    if not found_bold:
                        return False
    return True

    _SUSPECT_LABELS = {DocItemLabel.TEXT, DocItemLabel.LIST_ITEM}
    _pdf = fitz.open(pdf_path)
    suspect_rows   = []
    element_coords = []

    for item, _level in doc.iterate_items():
        if item.label in (
            DocItemLabel.TEXT, DocItemLabel.FORMULA, DocItemLabel.PICTURE,
            DocItemLabel.TABLE, DocItemLabel.LIST_ITEM, DocItemLabel.CODE,
        ) and item.prov:
            prov = item.prov[0]
            bbox = prov.bbox
            element_coords.append({
                "type": item.label.value, "page": prov.page_no,
                "bbox_l": round(bbox.l, 1), "bbox_t": round(bbox.t, 1),
                "bbox_r": round(bbox.r, 1), "bbox_b": round(bbox.b, 1),
                "text": getattr(item, "text", "")[:120],
            })

        if item.label not in _SUSPECT_LABELS or not item.prov:
            continue
        raw_text = getattr(item, "text", "")
        if "\n" in raw_text or len(raw_text) > 200:
            continue

        prov = item.prov[0]
        page = _pdf[prov.page_no - 1]
        bbox = prov.bbox
        rect = fitz.Rect(bbox.l, page.rect.height - bbox.t, bbox.r, page.rect.height - bbox.b)
        blocks = page.get_text("dict", clip=rect)["blocks"]
        if sum(len(b.get("lines", [])) for b in blocks) != 1:
            continue

        info = dominant_span_info(page, bbox, page.rect.height)
        if info is None or not info["bold"]: continue
        if (info["base_font"], info["size"]) not in heading_signatures: continue
        if not bold_not_preceded_by_normal(page, bbox, page.rect.height): continue

        suspect_rows.append({
            "label": item.label.value, "page": prov.page_no, "font": info["font"],
            "base_font": info["base_font"], "size": info["size"], "text": raw_text[:80],
            "bbox": (round(bbox.l,1), round(bbox.t,1), round(bbox.r,1), round(bbox.b,1)),
        })
    _pdf.close()
    return element_coords, suspect_rows

1.7 cell mới

In [ ]:
def merge_and_normalize_headings(rows, suspect_rows):
    """Gộp, gán level cho heading rác, loại trùng và sắp xếp"""
    size_to_level = {r["font_size"]: r["level"] for r in rows}
    normalized = []

    for r in rows:
        normalized.append({
            "level": r["level"], "text": r["text"], "page": r["page"],
            "font_size": r["font_size"], "bbox_l": r["bbox_l"], "bbox_t": r["bbox_t"],
            "bbox_r": r["bbox_r"], "bbox_b": r["bbox_b"], "source": "docling",
        })

    for r in suspect_rows:
        level = size_to_level.get(r["size"])
        if level is None:
            known = sorted(size_to_level.items(), key=lambda x: x[0])
            level = 1
            for sz, lv in known:
                if r["size"] >= sz: level = lv
        l, t, r_, b = r["bbox"]
        normalized.append({
            "level": level, "text": r["text"], "page": r["page"], "font_size": r["size"],
            "bbox_l": l, "bbox_t": t, "bbox_r": r_, "bbox_b": b, "source": "suspect",
        })

    seen, deduped = set(), []
    for r in normalized:
        key = (r["page"], r["text"].strip().lower()[:60])
        if key not in seen:
            seen.add(key)
            deduped.append(r)

    return sorted(deduped, key=lambda r: (r["page"], -r["bbox_t"]))

1.8 Tìm các heading trong file pdf mà bị docling nhận diện nhầm thành text

1.9 gộp các heading bị nhận diện nhầm ở 6 vào 5 theo đúng vị trí mà nó nên có và tạo ra được bảng các heading trong toàn file pdf

2   Thử kiểm tra xem file pdf bên trên có bookmark không, nếu không có thì tự tìm kiếm table of content/mục lục --- làm cái này bởi vì phải có cấu trúc cây thì mới dễ dàng phân vùng và tìm kiếm.

2.1 thư viện

2.2 nếu có bookmark tìm và kiểm tra bookmark và xuất mục lục thành dạng cây


In [ ]:
# ── Bước 3: Code xử lý ───────────────────────────────────────
from pypdf import PdfReader


def read_bookmarks(outline_items, reader, level=1):
    """Duyệt đệ quy danh sách bookmark, hỗ trợ lồng nhiều cấp."""
    result = []
    for item in outline_items:
        if isinstance(item, list):
            result.extend(read_bookmarks(item, reader, level + 1))
        else:
            title = item.title.strip() if item.title else "(không có tiêu đề)"
            page_num = None
            try:
                page_num = reader.get_destination_page_number(item) + 1
            except Exception:
                pass
            result.append({"level": level, "title": title, "page": page_num})
    return result


def extract_bookmark_tree(pdf_path):
    """Mở PDF và trích xuất toàn bộ bookmark thành cây phân cấp."""
    reader = PdfReader(pdf_path)

    if not reader.outline:
        print(" File PDF này không có bookmark.")
        print("   → Hãy dùng code extract_toc_tree (đọc từ trang Mục lục) thay thế.")
        return []

    print(f" Tìm thấy bookmark — Tổng số trang PDF: {len(reader.pages)}\n")
    return read_bookmarks(reader.outline, reader, level=1)


def print_bookmark_tree(tree):
    """In cây bookmark ra màn hình."""
    if not tree:
        return
    print("=" * 55)
    print(" CẤU TRÚC MỤC LỤC (từ Bookmark)")
    print("=" * 55)
    for item in tree:
        indent   = "  " * (item["level"] - 1)
        dash     = "─" * item["level"]
        page_str = f"  →  trang {item['page']}" if item["page"] else ""
        print(f"{indent}{dash} {item['title']}{page_str}")
    print("=" * 55)
    print(f"Tổng cộng: {len(tree)} mục")


def export_to_csv(tree, output_path="bookmark_toc.csv"):
    """Xuất kết quả ra file CSV."""
    import csv
    with open(output_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["level", "title", "page"])
        writer.writeheader()
        writer.writerows(tree)
    print(f" Đã xuất CSV: {output_path}")
    files.download(output_path)


def export_to_excel(tree, output_path="bookmark_toc.xlsx"):
    """Xuất kết quả ra file Excel."""
    import pandas as pd
    df = pd.DataFrame(tree)
    df.columns = ["Cấp độ", "Tiêu đề", "Số trang"]
    df.to_excel(output_path, index=False)
    print(f" Đã xuất Excel: {output_path}")
    files.download(output_path)


# # ── Bước 4: Chạy ─────────────────────────────────────────────
# tree = extract_bookmark_tree(pdf_path1111)

# if tree:
#     print_bookmark_tree(tree)

    # Bỏ comment nếu muốn tải file kết quả về máy:
    # export_to_csv(tree)
    # export_to_excel(tree)

2.3 nếu không có thì dùng hàm này để tìm các heading trong mục lục/table of content và kèm theo các đặc điểm của chúng - để lúc sau có thể chia nhóm và chia theo cấu trúc cây

In [ ]:
import re
import unicodedata
import pdfplumber
from collections import Counter

# --- Hằng số ---
TOC_KEYWORDS = ["mục lục", "table of contents", "contents", "content", "m ục l ục", "m u c l u c", "mục  lục", "m ục  l ục"]
TOP_MARGIN = 70
BOT_MARGIN = 60
FONT_SIZE_DIFF_THRESHOLD = 1.5


def clean_toc_text(text):
    if not text:
        return ""
    text = re.sub(r'[\s\.\-\–\_,\:]+$', '', text)
    return text.strip()

def normalize_toc_line(text):
    text = unicodedata.normalize('NFC', text.strip().lower())
    text = " ".join(text.split())
    return text

def is_roman(s):
    if not s: return False
    return bool(re.match(r'^(X{0,3})(IX|IV|V?I{0,3})$', s))

def is_arabic(s):
    return bool(re.fullmatch(r'\d+', s.strip()))

def is_roman_strict(s):
    if not s: return False
    return bool(re.match(r'^(X{0,3})(IX|IV|V?I{0,3})$', s))

def type_of_token(w):
    if re.match(r'^\d+$', w): return 'ARABIC'
    if is_roman_strict(w): return 'ROMAN'
    if len(w) == 1 and w.isupper(): return 'LETTER'
    return 'UNKNOWN'

def is_so(w):
    w = w.strip('()')
    w = re.sub(r'[:.]+$', '', w)
    if not w: return False
    parts = w.split('.')
    for p in parts:
        if not (re.match(r'^\d+$', p) or (len(p)==1 and p.isupper()) or is_roman_strict(p)):
            return False
    return True

def get_index_and_group(line):
    words = line.strip().split()
    if not words: return None, None

    w0_clean = re.sub(r'[:.]+$', '', words[0])
    w1_clean = re.sub(r'[:.]+$', '', words[1]) if len(words) > 1 else ''

    idx = None
    if len(words) > 1 and not is_so(w0_clean) and is_so(w1_clean):
        idx = w0_clean + ' ' + w1_clean
    elif is_so(w0_clean):
        idx = w0_clean

    if not idx:
        return None, None

    idx_words = idx.split()
    if len(idx_words) == 2:
        group = f'PREFIX_{idx_words[0]}_{type_of_token(idx_words[1])}'
    else:
        parts = idx_words[0].split('.')
        group = '.'.join(type_of_token(p) for p in parts)

    return idx, group


def get_body_lines(page, top_margin, bot_margin):
    h = page.height
    w = page.width

    cropped = page.crop((0, top_margin, w, h - bot_margin))
    words = cropped.extract_words(
        extra_attrs=["size", "fontname"],
        y_tolerance=3,
    )
    if not words:
        return []

    lines = []
    current_line = [words[0]]
    for word in words[1:]:
        if abs(word['top'] - current_line[0]['top']) < 3:
            current_line.append(word)
        else:
            lines.append(current_line)
            current_line = [word]
    lines.append(current_line)

    result = []
    for line in lines:
        line_sorted = sorted(line, key=lambda w: w['x0'])
        full_text = " ".join(w['text'] for w in line_sorted).strip()

        page_num = None
        page_num_type = None
        heading_text_str = full_text

        m_dots_arabic = re.search(r'(.*?)(?:[\.\-\_]{3,}\s*|\s{4,})(\d+)\s*$', full_text)
        m_dots_roman  = re.search(r'(.*?)(?:[\.\-\_]{3,}\s*|\s{4,})([ivxlcdmIVXLCDM]{1,6})\s*$', full_text)

        if m_dots_arabic:
            page_num          = int(m_dots_arabic.group(2))
            page_num_type     = 'arabic'
            heading_text_str  = m_dots_arabic.group(1).strip()
        elif m_dots_roman:
            page_num          = m_dots_roman.group(2)
            page_num_type     = 'roman'
            heading_text_str  = m_dots_roman.group(1).strip()
        else:
            rightmost  = line_sorted[-1]
            token      = rightmost['text'].strip()

            if len(line_sorted) > 1:
                token_clean = re.sub(r'^[\.\-\_]+', '', token)
                if is_arabic(token_clean):
                    page_num          = int(token_clean)
                    page_num_type     = 'arabic'
                    heading_text_str  = " ".join(w['text'] for w in line_sorted[:-1]).strip()
                elif is_roman(token_clean):
                    page_num          = token_clean
                    page_num_type     = 'roman'
                    heading_text_str  = " ".join(w['text'] for w in line_sorted[:-1]).strip()

        heading_text_str = clean_toc_text(heading_text_str)

        heading_words = []
        if page_num is not None and len(line_sorted) > 1:
            last_text = line_sorted[-1]['text']
            if str(page_num) in last_text:
                heading_words = line_sorted[:-1]
            else:
                heading_words = line_sorted
        else:
            heading_words = line_sorted

        heading_words_clean = []
        for w in heading_words:
            if re.fullmatch(r'[\.\-\_]+', w['text']):
                continue
            heading_words_clean.append(w)

        if not heading_words_clean:
            heading_words_clean = heading_words

        if len(heading_words_clean) > 0:
            x0       = min(w['x0'] for w in heading_words_clean)
            y0       = heading_words_clean[0]['top']
            fonts    = [w.get("fontname", "") for w in heading_words_clean if w.get("fontname")]
            fontname = Counter(fonts).most_common(1)[0][0] if fonts else "unknown"

            idx, _ = get_index_and_group(heading_text_str)
            if idx:
                idx_word_count    = len(idx.split())
                after_index_words = heading_words_clean[idx_word_count:]
                index_words       = heading_words_clean[:idx_word_count]

                if after_index_words:
                    avg_size = after_index_words[0].get('size', 0)
                else:
                    sizes    = [w.get('size', 0) for w in heading_words_clean if w.get('size', 0) > 0]
                    avg_size = sum(sizes) / len(sizes) if sizes else 0

                index_size = index_words[0].get('size', 0) if index_words else avg_size
            else:
                avg_size   = heading_words_clean[0].get('size', 0)
                index_size = avg_size

        else:
            x0         = min(w['x0'] for w in line_sorted)
            y0         = line_sorted[0]['top']
            avg_size   = 0
            index_size = 0
            fontname   = "unknown"

        is_bold = "bold" in fontname.lower()

        result.append({
            "text":          heading_text_str,
            "x0":            x0,
            "y0":            y0,
            "avg_size":      avg_size,
            "index_size":    index_size,
            "fontname":      fontname,
            "is_bold":       is_bold,
            "font_weight":   700 if is_bold else 400,
            "page_num":      page_num,
            "page_num_type": page_num_type,
        })
    return result


def is_toc_title(line):
    text_clean = " ".join(line['text'].strip().lower().split())
    text_clean = unicodedata.normalize('NFC', text_clean)
    for kw in TOC_KEYWORDS:
        kw_norm = unicodedata.normalize('NFC', kw.lower())
        if kw_norm in text_clean:
            return True
    return False


def extract_toc_headings(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:

        toc_start_page     = None
        toc_start_line_idx = None

        for page_idx, page in enumerate(pdf.pages):
            if page_idx > 20:
                break
            lines = get_body_lines(page, TOP_MARGIN, BOT_MARGIN)
            for i, line in enumerate(lines):
                if is_toc_title(line):
                    toc_start_page     = page_idx
                    toc_start_line_idx = i + 1
                    print(f" Tìm thấy mục lục ở trang {page_idx + 1}, dòng: \"{line['text']}\"")
                    break
            if toc_start_page is not None:
                break

        if toc_start_page is None:
            print(" Không tìm thấy mục lục trong file này.")
            return []

        all_lines = []
        for page_idx in range(
            toc_start_page,
            min(toc_start_page + 10, len(pdf.pages))
        ):
            page_lines = get_body_lines(pdf.pages[page_idx], TOP_MARGIN, BOT_MARGIN)
            start_idx  = toc_start_line_idx if page_idx == toc_start_page else 0
            all_lines.extend(page_lines[start_idx:])

        toc_lines       = []
        toc_index       = {}
        last_valid_page = None

        for i, line in enumerate(all_lines):
            current_page      = line['page_num']
            current_page_type = line['page_num_type']

            cleaned_text = clean_toc_text(line['text'])
            if not cleaned_text:
                continue

            text_norm = normalize_toc_line(cleaned_text)
            if not text_norm:
                continue

            if current_page_type == 'arabic':
                if last_valid_page is not None and current_page < last_valid_page:
                    is_end = True
                    lookahead_count = 0
                    for j in range(i + 1, len(all_lines)):
                        next_page      = all_lines[j]['page_num']
                        next_page_type = all_lines[j]['page_num_type']
                        if next_page_type == 'arabic':
                            lookahead_count += 1
                            if next_page >= last_valid_page:
                                is_end = False
                                break
                            if lookahead_count >= 2:
                                break
                    if is_end:
                        print(f" Kết thúc TOC do số trang lùi "
                              f"(từ {last_valid_page} → {current_page}) "
                              f"tại: \"{cleaned_text}\"")
                        break
                last_valid_page = current_page

            elif current_page is None:
                if last_valid_page is not None:
                    has_future_page = False
                    for j in range(i + 1, min(i + 5, len(all_lines))):
                        next_page      = all_lines[j]['page_num']
                        next_page_type = all_lines[j]['page_num_type']
                        if next_page_type == 'arabic' and next_page >= last_valid_page:
                            has_future_page = True
                            break
                    if not has_future_page:
                        print(f" Kết thúc TOC do hết số trang tiếp nối, "
                              f"dừng trước: \"{cleaned_text}\"")
                        break

            if text_norm not in toc_index:
                toc_index[text_norm] = line['avg_size']
            else:
                toc_size     = toc_index[text_norm]
                curr_size    = line['avg_size']
                size_differs = (
                    toc_size > 0 and curr_size > 0
                    and abs(curr_size - toc_size) >= FONT_SIZE_DIFF_THRESHOLD
                )
                if size_differs:
                    print(f" Kết thúc TOC do lặp nội dung chính "
                          f"(font size khác) tại: \"{cleaned_text}\"")
                    break

            line['text'] = cleaned_text
            toc_lines.append(line)

        return toc_lines


# # ── Chạy ─────────────────────────────────────────────────────────────────────
# real_heading_muc_luc = extract_toc_headings(pdf_path)

# print(f"\nTổng heading trong mục lục: {len(real_heading_muc_luc)}")

# print("\n===== PHÂN TÍCH HEADING MỤC LỤC =====")
# print(f"{'STT':<5} {'x0 (lề)':<12} {'avg_size':<12} {'index_size':<12} {'Weight':<10} {'fontname':<35} {'index':<15} {'text'}")
# print("-" * 145)
# for i, item in enumerate(real_heading_muc_luc, 1):
#     idx, _ = get_index_and_group(item['text'])
#     print(f"{i:<5} {item['x0']:<12.2f} {item['avg_size']:<12.2f} {item['index_size']:<12.2f} {item['font_weight']:<10} {item['fontname']:<35} {(idx or ''):<15} {item['text']}")

2.4 trong heading thì có những heading có chỉ số và có những cái không có chỉ số thì đoạn code này sẽ gồm các hàm định nghĩa chỉ số và tìm những heading có chỉ số và không có chỉ số sau đó chia phân nhóm các heading đó

In [ ]:
import re
from collections import defaultdict

# CÁC HÀM HỖ TRỢ XỬ LÝ CHỈ SỐ
def is_roman(s):
    if not s: return False
    return bool(re.match(r'^(X{0,3})(IX|IV|V?I{0,3})$', s))

def type_of_token(w):
    if re.match(r'^\d+$', w): return 'TOKEN'
    if is_roman(w): return 'TOKEN'
    if len(w) == 1 and w.isupper(): return 'TOKEN'
    return 'UNKNOWN'

def is_so(w):
    w = w.strip('()')
    w = re.sub(r'[:.]+$', '', w)
    if not w: return False
    parts = w.split('.')
    for p in parts:
        if not (re.match(r'^\d+$', p) or (len(p)==1 and p.isupper()) or is_roman(p)):
            return False
    return True

# BƯỚC 1: NHẬN DIỆN CHỈ SỐ
def get_index_and_group(line):
    words = line.strip().split()
    if not words: return None, None

    w0_clean = re.sub(r'[:.]+$', '', words[0])
    w1_clean = re.sub(r'[:.]+$', '', words[1]) if len(words) > 1 else ''

    idx = None
    if len(words) > 1 and not is_so(w0_clean) and is_so(w1_clean):
        idx = w0_clean + ' ' + w1_clean
    elif is_so(w0_clean):
        idx = w0_clean

    if not idx: return None, None

    idx_words = idx.split()
    if len(idx_words) == 2:
        group = f'PREFIX_{idx_words[0]}_{type_of_token(idx_words[1])}'
    else:
        parts = idx_words[0].split('.')
        group = '.'.join(type_of_token(p) for p in parts)

    return idx, group

# BƯỚC 2: GOM CÁI CÓ CHỈ SỐ VÀ CÁI KHÔNG CÓ CHỈ SỐ
def group_all_headings(heading_list):
    indexed_groups  = defaultdict(list)
    no_index_groups = defaultdict(list)

    if isinstance(heading_list, str): heading_list = [heading_list]
    elif isinstance(heading_list, dict): heading_list = [heading_list]

    for i, original_item in enumerate(heading_list):
        item = {'text': original_item} if isinstance(original_item, str) else original_item.copy()
        item['original_order'] = i

        text = item.get('text', '')
        if not text: continue

        idx, group_signature = get_index_and_group(text)

        if idx:
            item['extracted_index'] = idx
            indexed_groups[group_signature].append(item)
        else:
            left_margin    = item.get('x0', 0)
            font_size      = item.get('avg_size', 0)
            font_name      = item.get('fontname', 'UNKNOWN')

            rounded_margin = round(left_margin, 1)
            rounded_size   = round(font_size, 1)

            no_idx_sig     = f"NO_INDEX_Margin({rounded_margin})_Size({rounded_size})_Font({font_name})"
            no_index_groups[no_idx_sig].append(item)

    return indexed_groups, no_index_groups

# BƯỚC 3: XÉT TÍNH CHA CON (CÔNG TẮC RẼ NHÁNH)
def check_hierarchy(indexed_groups, no_index_groups, margin_tolerance=5):
    # Dấu hiệu 2: Bất kỳ cặp no_index nào CÙNG LỀ mà KHÁC font/size -> KHÔNG phân cấp
    no_idx_items_all = []
    for items in no_index_groups.values():
        no_idx_items_all.extend(items)

    for i, item_a in enumerate(no_idx_items_all):
        for item_b in no_idx_items_all[i+1:]:
            if abs(item_a.get('x0', 0) - item_b.get('x0', 0)) <= margin_tolerance:
                if (item_a.get('fontname') != item_b.get('fontname') or
                    round(item_a.get('avg_size', 0), 1) != round(item_b.get('avg_size', 0), 1)):
                    return False

    # Mặc định: có phân bậc
    return True

# BƯỚC 4: GOM NHÓM THEO FONT VÀ SIZE (Nếu mục lục phẳng)
def group_by_font_and_size(indexed_groups, no_index_groups):
    merged_groups = defaultdict(list)
    def add_to_merged(groups_dict):
        for sig, items in groups_dict.items():
            for item in items:
                fontname = item.get('fontname', 'UNKNOWN')
                avg_size = round(item.get('avg_size', 0), 1)
                merged_groups[f"Font({fontname})_Size({avg_size})"].append(item)
    add_to_merged(indexed_groups)
    add_to_merged(no_index_groups)
    return dict(merged_groups)

# BƯỚC 5: XỬ LÝ "CHỈ SỐ GIẢ"
# BƯỚC 5: Thêm cờ đánh dấu khi gộp
def merge_fake_indexed_to_no_index(indexed_groups, no_index_groups, margin_tolerance=5):
    def is_no_dot(group_sig):
        if group_sig == 'TOKEN': return True   # Thay cho ARABIC/ROMAN/LETTER
        if re.match(r'^PREFIX_\w+_TOKEN$', group_sig): return True
        return False

    remaining_indexed = {}
    for group_sig, indexed_items in indexed_groups.items():
        if not is_no_dot(group_sig):
            remaining_indexed[group_sig] = indexed_items
            continue

        unmatched = []
        for item in indexed_items:
            margin   = item.get('x0', 0)
            fontname = item.get('fontname', 'UNKNOWN')
            avg_size = round(item.get('avg_size', 0), 1)

            matched_sig = None
            for no_idx_sig, no_items in no_index_groups.items():
                if not no_items: continue
                rep_margin = no_items[0].get('x0', 0)
                if (abs(margin - rep_margin) <= margin_tolerance
                        and fontname == no_items[0].get('fontname', 'UNKNOWN')
                        and avg_size == round(no_items[0].get('avg_size', 0), 1)):
                    matched_sig = no_idx_sig
                    break

            if matched_sig:
                item['_merged_by_step5'] = True   # ← Đánh dấu: Bước 5 đã xác nhận là giả
                no_index_groups[matched_sig].append(item)
            else:
                unmatched.append(item)

        if unmatched: remaining_indexed[group_sig] = unmatched

    return remaining_indexed, no_index_groups

# BƯỚC 6: TRỤC XUẤT CÁC PHẦN TỬ LẠC QUẺ
def split_mixed_category_groups(indexed_groups, no_index_groups):
    def get_item_category(item):
        text = item.get('text', '')
        idx, group_sig = get_index_and_group(text)
        if not idx: return 'NO_INDEX', None
        if group_sig.startswith('PREFIX_'): return 'PREFIX', group_sig
        if '.' in group_sig: return 'MULTI_LEVEL', group_sig
        if group_sig == 'TOKEN': return 'SIMPLE', group_sig   # Thay cho 3 cái cũ
        return 'UNKNOWN', group_sig

    new_no_index = {}
    for sig, items in no_index_groups.items():
        no_index_items = []
        indexed_by_cat = defaultdict(list)

        for item in items:
            cat, _ = get_item_category(item)
            if cat == 'NO_INDEX': no_index_items.append(item)
            else: indexed_by_cat[cat].append(item)

        if len(indexed_by_cat) <= 1:
            new_no_index[sig] = items
            continue

        first_order = {c: min(i.get('original_order', float('inf')) for i in l) for c, l in indexed_by_cat.items()}
        sorted_cats = sorted(indexed_by_cat.keys(), key=lambda c: first_order[c])
        primary_cat = sorted_cats[0]
        new_no_index[sig] = no_index_items + indexed_by_cat[primary_cat]

        for cat in sorted_cats[1:]:
            for item in indexed_by_cat[cat]:
                item['_is_evicted'] = True
                item['_evicted_from'] = sig

                _, orig_sig = get_index_and_group(item.get('text', ''))
                if orig_sig:
                    if orig_sig not in indexed_groups: indexed_groups[orig_sig] = []
                    indexed_groups[orig_sig].append(item)

    return indexed_groups, new_no_index

# BƯỚC 7: BẮT LỖI LOGIC NGƯỢC ĐỜI (ARABIC, LETTER, ROMAN CHUNG BẬC)
def get_hierarchy_level(group_sig):
    if group_sig in ('LETTER', 'ROMAN', 'ARABIC') or group_sig.startswith('PREFIX_'): return 1
    if '.' in group_sig: return 2
    return 99

def execute_reverse_hierarchy_check(indexed_groups, no_index_groups):
    # Gốc so: LẤY TỪ CẢ HAI RỔ — bất kỳ item nào còn extracted_index đều dùng làm mốc
    all_indexed_entries = []
    for sig, items in indexed_groups.items():
        for item in items:
            all_indexed_entries.append((sig, item))

    # Thêm các item trong no_index mà vẫn còn chỉ số gốc
    for sig, items in no_index_groups.items():
        for item in items:
            if item.get('extracted_index'):
                _, orig_sig = get_index_and_group(item.get('text', ''))
                if orig_sig:
                    all_indexed_entries.append((orig_sig, item))

    fakes = []
    for sig_loai, item in all_indexed_entries:
        if not item.get('_is_evicted'): continue

        bac_loai = get_hierarchy_level(sig_loai)
        le_loai = item.get('x0', 0)

        is_fake = False
        for sig_valid, valid_item in all_indexed_entries:
            if valid_item is item or valid_item.get('_is_evicted'): continue

            bac_valid = get_hierarchy_level(sig_valid)
            le_valid = valid_item.get('x0', 0)

            if bac_loai <= bac_valid and le_loai > le_valid:
                is_fake = True
                break

        if is_fake: fakes.append((sig_loai, item))

    for sig_loai, item in fakes:
        if sig_loai in indexed_groups and item in indexed_groups[sig_loai]:
            indexed_groups[sig_loai].remove(item)

        original_no_idx_sig = item.get('_evicted_from')
        if original_no_idx_sig and original_no_idx_sig in no_index_groups:
            no_index_groups[original_no_idx_sig].append(item)
        else:
            margin, avg_size = item.get('x0', 0), round(item.get('avg_size', 0), 1)
            font_name = item.get('fontname', 'UNKNOWN')
            no_idx_sig = f"NO_INDEX_Margin({round(margin, 1)})_Size({avg_size})_Font({font_name})"
            no_index_groups[no_idx_sig].append(item)

        if 'extracted_index' in item:
            del item['extracted_index']

    empty_keys = [k for k, v in indexed_groups.items() if not v]
    for k in empty_keys: del indexed_groups[k]

    return indexed_groups, no_index_groups


# HÀM IN KẾT QUẢ (TUYỆT ĐỐI KHÔNG GỘP BỪA BÃI)

def analyze_and_print_parent_child(indexed_groups, no_index_groups, margin_tolerance=5):
    def get_rep(items):
        if not items: return None, None, None
        return (items[0].get('x0', 0), items[0].get('fontname', 'UNKNOWN'), round(items[0].get('avg_size', 0), 1))

    def print_block(sig, items):
        if not items: return
        margin, fontname, avg_size = get_rep(items)
        print(f"\n[ Nhóm: {sig} | Lề: {round(margin, 1)} | Font: {fontname} | Size: {avg_size} ]")
        for item in items:
            idx  = item.get('extracted_index', '')
            print(f"  + {idx:<10} | {item.get('text', '')}")

    # In thẳng từng nhóm đã được xử lý ở các bước trước, không gộp gì thêm
    for sig, items in indexed_groups.items():
        print_block(sig, items)

    for sig, items in no_index_groups.items():
        print_block(sig, items)


# PIPELINE TỔNG THỂ
def process_toc_v2(real_heading_muc_luc):
    indexed_toc, no_index_toc = group_all_headings(real_heading_muc_luc)
    has_hierarchy = check_hierarchy(indexed_toc, no_index_toc, margin_tolerance=5)

    print("===== MỤC LỤC TỔNG HỢP (PHIÊN BẢN CHÍNH THỨC) =====")

    if not has_hierarchy:
        final_groups = group_by_font_and_size(indexed_toc, no_index_toc)
        analyze_and_print_parent_child(final_groups, {}, margin_tolerance=5)

        # grouped_heading chính là final_groups
        grouped_heading = final_groups.copy()

        return final_groups, {}, grouped_heading
    else:
        indexed_toc, no_index_toc = merge_fake_indexed_to_no_index(indexed_toc, no_index_toc, margin_tolerance=5)
        indexed_toc, no_index_toc = split_mixed_category_groups(indexed_toc, no_index_toc)
        indexed_toc, no_index_toc = execute_reverse_hierarchy_check(indexed_toc, no_index_toc)

        analyze_and_print_parent_child(indexed_toc, no_index_toc, margin_tolerance=5)

        # TẠO BIẾN grouped_heading: Gộp chung cả 2 rổ lại thành 1 dictionary duy nhất
        grouped_heading = {}
        grouped_heading.update(indexed_toc)
        grouped_heading.update(no_index_toc)

        return indexed_toc, no_index_toc, grouped_heading



2.5 sắp xếp các nhóm đã tìm được ở bước trên và tạo cấu trúc cây và bậc cho chúng

giải thích logic

b1: tìm dòng nào là dòng "mục lục", "table of content" hoặc "content" có trong struct_merged (output của 1.7) - toàn bộ heading trong pdf
b2: tìm dòng ngay sau dòng mục lục đó , đi so sánh trong real_heading_muc_luc rồi gộp các heading từ dòng ngay sau đó thành behind_muc_luc_real_heading_muc_luc
b3: đi tìm bậc trong behind_muc.......bằng cách xem nhóm nào có heading được xếp trước trong behind....

In [ ]:
import re
from difflib import SequenceMatcher

# Hàm hỗ trợ tính phần trăm giống nhau giữa 2 chuỗi
def similar(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

# BƯỚC 8: TÌM CỘT MỐC "MỤC LỤC" VÀ GÁN BẬC DỰA VÀO ĐÓ

def get_start_idx_after_toc(struct_merged, real_heading_muc_luc):
    """
    Tìm chữ 'Mục lục' -> Lấy dòng liền sau -> Đối chiếu xuống real_heading (độ giống >= 80%).
    """
    # 1. Tìm vị trí dòng chỉ có từ mục lục / table of content / contents
    toc_idx = -1
    for i, row in enumerate(struct_merged):
        txt = row.get('Text', row.get('text', '')).strip().lower()
        if txt in ['mục lục', 'mucluc', 'table of content', 'table of contents', 'contents']:
            toc_idx = i
            break

    # 2. Lấy text dòng ngay phía sau dòng "mục lục"
    if toc_idx != -1 and toc_idx + 1 < len(struct_merged):
        candidate_text = struct_merged[toc_idx + 1].get('Text', struct_merged[toc_idx + 1].get('text', '')).strip()

        # 3. Xét trong real_heading_muc_luc xem dòng nào tương đồng >= 80%
        for k, item in enumerate(real_heading_muc_luc):
            r_text = item.get('text', '').strip() if isinstance(item, dict) else item.strip()

            if similar(candidate_text, r_text) >= 0.8:
                return k # Lấy vị trí này làm mốc

    # Fallback: Nếu không tìm thấy, mặc định lấy từ đầu danh sách
    return 0


def assign_levels_and_build_tree(grouped_heading, real_heading_muc_luc, struct_merged):
    """
    Gán bậc cho từng nhóm và xuất ra list data + cấu trúc cây dạng Text.
    """
    # 1. Tìm cột mốc chia cắt
    start_idx = get_start_idx_after_toc(struct_merged, real_heading_muc_luc)

    all_groups = []

    # 2. Xét từng nhóm đã gom để tìm ra độ ưu tiên
    for sig, items in grouped_heading.items():
        if not items: continue

        # Chỉ xét những item nằm trong phần behind_muc_luc (original_order >= start_idx)
        items_behind_toc = [item for item in items if item.get('original_order', -1) >= start_idx]

        if items_behind_toc:
            # Nhóm có xuất hiện sau chữ Mục Lục -> Ưu tiên dựa vào vị trí sớm nhất
            first_order_behind = min(item.get('original_order') for item in items_behind_toc)
        else:
            # Nhóm KHÔNG CÓ mặt sau Mục Lục -> Đẩy xuống bét (gán vô cực)
            first_order_behind = float('inf')

        absolute_first_order = min(item.get('original_order', float('inf')) for item in items)

        all_groups.append({
            'signature': sig,
            'first_order_behind': first_order_behind,
            'absolute_first_order': absolute_first_order,
            'items': items
        })

    # 3. Sắp xếp các nhóm để phân bậc (càng nhỏ càng ưu tiên)
    all_groups.sort(key=lambda g: (g['first_order_behind'], g['absolute_first_order']))

    # 4. Gán số Bậc (Level)
    order_to_level = {}
    current_level = 1
    for group in all_groups:
        for item in group['items']:
            order_to_level[item.get('original_order')] = current_level
        current_level += 1

    # BIẾN OUTPUT 1: real_heading_muc_luc_level
    real_heading_muc_luc_level = []
    for i, original_item in enumerate(real_heading_muc_luc):
        item_copy = original_item.copy() if isinstance(original_item, dict) else {'text': original_item}
        item_copy['level'] = order_to_level.get(i, 1)

        if 'extracted_index' not in item_copy:
            idx, _ = get_index_and_group(item_copy.get('text', ''))
            item_copy['extracted_index'] = idx if idx else ''

        real_heading_muc_luc_level.append(item_copy)

    # BIẾN OUTPUT 2: heading_muc_luc_tree
    tree_lines = []
    for item in real_heading_muc_luc_level:
        level = item.get('level', 1)
        text = item.get('text', '')

        # Bậc 1 thụt 0, Bậc 2 thụt 4 khoảng trắng, Bậc 3 thụt 8...
        indent = "    " * (level - 1)
        tree_lines.append(f"{indent}- {text}")

    heading_muc_luc_tree = "\n".join(tree_lines)

    # Trả về các biến đầu ra
    return real_heading_muc_luc_level, heading_muc_luc_tree, start_idx


# HÀM IN TEST KẾT QUẢ
def print_final_output(real_heading_muc_luc_level, heading_muc_luc_tree, start_idx):
    print("\n" + "="*80)
    print(" DANH SÁCH HEADING NẰM SAU MỤC LỤC (BEHIND MỤC LỤC) ".center(80, '='))
    print("="*80)
    behind_list = real_heading_muc_luc_level[start_idx:]
    if not behind_list:
        print("(Không có heading nào sau Mục lục / Hoặc không tìm thấy chữ Mục lục)")
    else:
        for i, item in enumerate(behind_list):
            real_stt = i + start_idx + 1
            text = item.get('text', '')
            print(f"[STT {real_stt:02d}] {text}")

    print("\n" + "="*80)
    print(" BẢNG DỮ LIỆU ĐÃ GẮN BẬC (LEVEL) ".center(80, '='))
    print("="*80)
    print(f"{'STT':<5} | {'Bậc':<4} | {'Chỉ số':<8} | {'Nội dung'}")
    print("-" * 110)
    for i, item in enumerate(real_heading_muc_luc_level, 1):
        level = item.get('level', '?')
        idx = item.get('extracted_index', '')
        text = item.get('text', '')
        print(f"{i:<5} | {level:<4} | {idx:<8} | {text}")

    print("\n" + "="*80)
    print(" CẤU TRÚC CÂY MỤC LỤC ".center(80, '='))
    print("="*80)
    print(heading_muc_luc_tree)


# # =========================================================
# # CÚ PHÁP ĐỂ BẠN GỌI VÀ CHẠY
# # =========================================================
# # Giả sử phía trên đã chạy: idx, no_idx, grouped_heading = process_toc_v2(real_heading_muc_luc)

# # Gọi hàm sinh data (Hứng 2 biến theo đúng yêu cầu của bạn)
# real_heading_muc_luc_level, heading_muc_luc_tree, start_idx = assign_levels_and_build_tree(
#     grouped_heading, real_heading_muc_luc, struct_merged
# )

# # Gọi hàm in để check bằng mắt
# print_final_output(real_heading_muc_luc_level, heading_muc_luc_tree, start_idx)

        2.6 soát lại 1 lượt các heading trong toàn file pdf đã tìm được được lưu trong struct_merged xem những heading nào bị trùng với các font monospace

In [ ]:
import fitz

def filter_monospace_headings(struct_merged, element_coords, pdf_path):
    """Lọc bỏ các heading có font monospace (từ 5 đoạn CODE đầu tiên + 10 font mặc định)"""
    code_fonts = set()
    code_count = 0
    _pdf_font = fitz.open(pdf_path)

    class _BBox:
        def __init__(self, l, t, r, b): self.l, self.t, self.r, self.b = l, t, r, b

    for e in element_coords:
        if e["type"] == "code":
            page = _pdf_font[e["page"] - 1]
            bbox = _BBox(e["bbox_l"], e["bbox_t"], e["bbox_r"], e["bbox_b"])
            info = dominant_span_info(page, bbox, page.rect.height)
            if info:
                code_fonts.add(info["base_font"].lower())
            code_count += 1
            if code_count >= 5:
                break

    monospace_fonts = {
        "courier", "consolas", "monaco", "inconsolata", "lucida console",
        "fira code", "source code pro", "menlo", "roboto mono", "jetbrains mono"
    }
    forbidden_fonts = code_fonts | monospace_fonts

    filtered_merged = []
    removed_headings = []   # ← MỚI: lưu lại các heading bị loại
    removed_count = 0

    for r in struct_merged:
        page = _pdf_font[r["page"] - 1]
        bbox = _BBox(r["bbox_l"], r["bbox_t"], r["bbox_r"], r["bbox_b"])
        info = dominant_span_info(page, bbox, page.rect.height)
        if info:
            bfont = info["base_font"].lower()
            is_forbidden = any(f in bfont for f in forbidden_fonts)
            if not is_forbidden:
                filtered_merged.append(r)
            else:
                removed_headings.append(r)   # ← lưu lại thay vì chỉ in ra
                removed_count += 1
        else:
            filtered_merged.append(r)

    print(f"Tổng kết: {len(filtered_merged)} giữ lại, {removed_count} bị loại")

    _pdf_font.close()
    return filtered_merged, removed_headings   # ← trả về thêm removed_headings

2.7 hàm cứu lại các heading bị loại bỏ ra khỏi headng bởi hàm trong 2.6 bên trên

In [ ]:
import fitz
from docling_core.types.doc import DocItemLabel


def resolve_removed_headings(struct_merged_filtered, removed_headings, element_coords):
    """
    Xử lý các heading đã bị loại bỏ (ví dụ do trùng font monospace):
      - Tìm element đứng NGAY SAU heading đó trong element_coords (cùng trang,
        gần nhất theo vị trí đọc: từ trên xuống, trái sang phải).
      - Nếu element đó là TEXT hoặc CODE -> gộp (union) bbox của heading vào bbox
        của element đó (mở rộng bbox của element để bao trọn cả heading).
      - Nếu element đó không phải TEXT/CODE (hoặc không tìm thấy) -> heading đó
        tự trở thành 1 item loại TEXT độc lập, thêm vào element_coords.

    Trả về:
      - element_coords đã được cập nhật (list mới, không sửa in-place)
      - removed_headings_log: log chi tiết xử lý từng heading bị loại
    """

    def reading_order_key(e):
        # Thứ tự đọc: trang tăng dần, rồi từ trên (t lớn) xuống dưới (t nhỏ), trái->phải
        return (e["page"], -e["bbox_t"], e["bbox_l"])

    # Sắp xếp element_coords theo thứ tự đọc để tìm "ngay sau" chính xác
    sorted_elements = sorted(element_coords, key=reading_order_key)

    new_element_coords = list(element_coords)  # copy, không sửa list gốc
    removed_headings_log = []

    for h in removed_headings:
        h_page = h["page"]
        h_bbox_t = h["bbox_t"]

        # Tìm element đứng ngay sau heading: cùng trang, có bbox_t nhỏ hơn
        # (tức nằm thấp hơn trên trang) và gần heading nhất
        candidates = [
            e for e in sorted_elements
            if e["page"] == h_page and e["bbox_t"] < h_bbox_t
        ]

        next_elem = None
        if candidates:
            # Gần nhất = bbox_t lớn nhất trong số các candidate (thấp hơn heading nhưng cao nhất trong nhóm đó)
            next_elem = max(candidates, key=lambda e: e["bbox_t"])

        if next_elem is not None and next_elem["type"] in (
            DocItemLabel.TEXT.value, DocItemLabel.CODE.value,
        ):
            # ── Gộp (union) bbox của heading vào bbox của next_elem ──
            old_bbox = (
                next_elem["bbox_l"], next_elem["bbox_t"],
                next_elem["bbox_r"], next_elem["bbox_b"],
            )

            next_elem["bbox_l"] = min(next_elem["bbox_l"], h["bbox_l"])
            next_elem["bbox_r"] = max(next_elem["bbox_r"], h["bbox_r"])
            next_elem["bbox_t"] = max(next_elem["bbox_t"], h["bbox_t"])
            next_elem["bbox_b"] = min(next_elem["bbox_b"], h["bbox_b"])

            removed_headings_log.append({
                "heading_text": h["text"], "page": h_page,
                "action": "merged_into_next", "merged_into_type": next_elem["type"],
                "old_bbox": old_bbox,
                "new_bbox": (next_elem["bbox_l"], next_elem["bbox_t"],
                             next_elem["bbox_r"], next_elem["bbox_b"]),
            })
        else:
            # ── Không đứng trước TEXT/CODE -> tự thành 1 item TEXT độc lập ──
            new_text_item = {
                "type": DocItemLabel.TEXT.value,
                "page": h_page,
                "bbox_l": h["bbox_l"], "bbox_t": h["bbox_t"],
                "bbox_r": h["bbox_r"], "bbox_b": h["bbox_b"],
                "text": h["text"],
            }
            new_element_coords.append(new_text_item)

            removed_headings_log.append({
                "heading_text": h["text"], "page": h_page,
                "action": "converted_to_text",
                "next_elem_type": next_elem["type"] if next_elem else None,
            })

    return new_element_coords, removed_headings_log

2.7 cell mới

In [ ]:
import difflib

def align_bookmark_with_toc(bookmark_tree, toc_items_list):
    """Đối chiếu Bookmark với TOC bằng thuật toán cửa sổ trượt (Sliding Window)"""
    real_heading_muc_luc_level = []
    last_matched_idx = 0

    for item in bookmark_tree:
        norm_title = "".join(str(item["title"]).split()).lower()
        best_idx, best_ratio, best_k = "", 0.0, last_matched_idx
        window_end = min(last_matched_idx + 15, len(toc_items_list))

        for k in range(last_matched_idx, window_end):
            toc_item = toc_items_list[k]
            norm_toc_text = "".join(str(toc_item.get("text", "")).split()).lower()
            ratio = difflib.SequenceMatcher(None, norm_title, norm_toc_text).ratio()
            if ratio > best_ratio:
                best_ratio, best_idx, best_k = ratio, toc_item.get("extracted_index", ""), k

        if best_ratio >= 0.5:
            idx = best_idx
            last_matched_idx = best_k + 1
        else:
            idx = ""

        real_heading_muc_luc_level.append({
            "text": item["title"], "level": item["level"],
            "page_num": item["page"], "extracted_index": idx,
        })

    tree_lines = ["    " * (item["level"] - 1) + "- " + item["title"] for item in bookmark_tree]
    heading_muc_luc_tree = "\n".join(tree_lines)

    return real_heading_muc_luc_level, heading_muc_luc_tree

2.8 hàm xác định trùng lặp heading giữa struct_merged với real_heading_muc_luc_level, thêm 1 cột bậc cho struct_merged , những heading mà cả 2 trùng lặp thì bậc của heading trong struct_merged cũng có bậc giống với heading trong real_headaing_muc_luc_level

In [ ]:
# def sync_levels_with_toc(struct_merged, toc_levels):
#     """
#     Đồng bộ bậc (level) cho các heading trong struct_merged dựa trên mục lục.
#     """
#     # 1. Tạo dictionary mapping (Ghép extracted_index và text lại với nhau)
#     toc_map = {}
#     for item in toc_levels:
#         if "text" in item:
#             # Lấy index (nếu có)
#             idx = item.get("extracted_index", "").strip()
#             # Nếu có index thì ghép vào, ví dụ: "0.1" + " " + "Mục đích" = "0.1 Mục đích"
#             full_text = f"{idx} {item['text']}".strip().lower() if idx else item["text"].strip().lower()

#             toc_map[full_text] = item["level"]

#     # 2. Tìm vị trí của tiêu đề "Mục lục"
#     toc_idx = -1
#     keywords = {"mục lục", "table of contents", "contents", "mục lục sơ bộ"}
#     for i, r in enumerate(struct_merged):
#         if r.get("text", "").strip().lower() in keywords:
#             toc_idx = i
#             break

#     # 3. Duyệt và cập nhật bậc cho các heading xếp sau mục lục
#     start_idx = toc_idx + 1 if toc_idx != -1 else 0
#     for i in range(start_idx, len(struct_merged)):
#         txt = struct_merged[i].get("text", "").strip().lower()
#         if txt in toc_map:
#             # Ghi đè level chuẩn từ Mục lục
#             struct_merged[i]["level"] = toc_map[txt]
#             # Thêm biến này để lệnh Print in ra được cấp bậc thay vì dấu "-"
#             struct_merged[i]["toc_level"] = toc_map[txt]
#             struct_merged[i]["is_toc_matched"] = True
#         else:
#             struct_merged[i]["is_toc_matched"] = False

#     return struct_merged

2.9 hàm tìm các heading bậc có dạng giống với heading bậc nhỏ nhất trong heading_muc_luc_tree, gán cho các heading tìm được có bậc nhỏ hơn bậc của heading bậc nhỏ nhất đó 1 bậc

In [ ]:
# def update_subheading_levels(struct_merged, real_heading_muc_luc_level):
#     import re
#     toc_indices = {}
#     for item in real_heading_muc_luc_level:
#         idx_str = item.get("extracted_index", "")
#         if idx_str and re.fullmatch(r'\d+(?:\.\d+)*', idx_str):
#             toc_indices[idx_str] = item.get("level", 1)

#     for r in struct_merged:
#         if r.get("is_toc_matched"):
#             continue
#         text = r.get("text", "")
#         m = re.match(r'^\s*(\d+(?:\.\d+)*)\.?\s+', text)
#         if m:
#             struct_idx = m.group(1)
#             best_prefix = None
#             for toc_idx in toc_indices:
#                 if struct_idx == toc_idx:
#                     best_prefix = toc_idx
#                     break
#                 elif struct_idx.startswith(toc_idx + "."):
#                     if best_prefix is None or len(toc_idx) > len(best_prefix):
#                         best_prefix = toc_idx
#             if best_prefix:
#                 level_diff = struct_idx.count(".") - best_prefix.count(".")
#                 if level_diff > 0:
#                     r["level"] = toc_indices[best_prefix] + level_diff
#                     r["is_sub_updated"] = True
#                     continue

#         # Không được TOC match, không được update → gán None
#         r["level"] = None

#     return struct_merged

2.10 không dùng 2 hàm trong 2.9 và 2.10 nữa thay vào đó là dùng hàm này để

**gán đúng cấp bậc (level) cho tất cả các heading được tìm thấy trong file PDF**

+ Nhận diện Heading chính (Khớp >= 70%): Nó so sánh từng heading trong PDF với danh sách Mục lục. Nếu giống nhau từ 70% trở lên, nó sẽ gán luôn level của mục lục cho heading đó. Sau khi khớp xong, nó "rút" luôn mục đó ra để các heading phía sau không bị nhận nhầm nữa (cơ chế chống trùng lặp).

+ Nhận diện Heading con (Dựa vào dấu chấm): Với những heading không nằm trong Mục lục (thường là các tiểu mục nhỏ hơn như 1.1.1, trong khi mục lục chỉ ghi tới 1.1), nó sẽ tự động soi chỉ số. Nếu thấy chỉ số con dài hơn chỉ số cha 1 dấu chấm, nó sẽ tự tính toán để gán level của con = level của cha + 1.

+ Loại bỏ Heading rác: Bất kỳ câu chữ nào bị nhận nhầm là heading mà không thỏa mãn 2 điều kiện trên (không có trong mục lục, cũng không có cấu trúc chỉ số hợp lệ) thì nó gán level = None.

In [ ]:
import difflib
import re

def similar(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()

def assign_levels_to_struct_merged(struct_merged, real_heading_muc_luc_level):
    # CHUẨN BỊ DỮ LIỆU
    toc_indices = {}
    available_tocs = []

    for item in real_heading_muc_luc_level:
        level = item.get("level", 1)
        idx_str = item.get("extracted_index", "").strip()
        raw_text = item.get("text", "").strip()

        # Gắn lại chỉ số vào text giống như dạng của struct_merged
        if idx_str:
            full_text = f"{idx_str} {raw_text}".strip().lower()
        else:
            full_text = raw_text.lower()

        available_tocs.append({
            "full_text": full_text,
            "level": level
        })

        # Lưu lại index để phục vụ Bước 2 (so dấu chấm)
        if idx_str and re.fullmatch(r'\d+(?:\.\d+)*', idx_str):
            toc_indices[idx_str] = level

    # DUYỆT TỪNG HEADING TRONG FILE PDF
    for r in struct_merged:
        txt_merged = r.get("text", "").strip().lower()

        # BƯỚC 1: Tìm nhóm khớp với mục lục (độ giống >= 70%)
        best_ratio = 0
        best_match_idx = -1

        # Rà từ trái sang phải trong danh sách TOC còn lại
        for i, toc_item in enumerate(available_tocs):
            ratio = similar(txt_merged, toc_item["full_text"])
            if ratio > best_ratio:
                best_ratio = ratio
                best_match_idx = i

        if best_ratio >= 0.70:
            r["level"] = available_tocs[best_match_idx]["level"]
            # CHỐT: Đã khớp thì "rút" luôn mục lục này ra để không bị so trùng nữa
            available_tocs.pop(best_match_idx)
            continue

        # BƯỚC 2: Tìm tiểu mục con và tính level dựa trên dấu chấm
        m = re.match(r'^\s*(\d+(?:\.\d+)*)\.?\s+', r.get("text", ""))
        if m:
            struct_idx = m.group(1)
            best_prefix = None

            # Tìm heading cha trong Mục lục
            for toc_idx in toc_indices:
                if struct_idx == toc_idx:
                    best_prefix = toc_idx
                    break
                elif struct_idx.startswith(toc_idx + "."):
                    if best_prefix is None or len(toc_idx) > len(best_prefix):
                        best_prefix = toc_idx

            if best_prefix:
                dots_struct = struct_idx.count(".")
                dots_prefix = best_prefix.count(".")
                level_diff = dots_struct - dots_prefix

                if level_diff > 0:
                    r["level"] = toc_indices[best_prefix] + level_diff
                    continue

        # BƯỚC 3: Không hợp lệ -> Gán None
        r["level"] = None

    return struct_merged

2.11 hàm này để xác định các heading còn sót level ở hàm trước trong struct_merged --- nó kiểu những "heading tự do"

In [ ]:
def get_missing_level_blocks(struct_merged):
    """
    Tìm và gom nhóm các heading bị sót (level=None) nằm kẹp giữa 2 heading hợp lệ.
    Chỉ xét các heading đằng sau Mục lục.
    """
    # 1. Tìm vị trí kết thúc của Mục lục
    toc_idx = -1
    keywords = {"mục lục", "table of contents", "contents", "mục lục sơ bộ"}
    for i, r in enumerate(struct_merged):
        if r.get("text", "").strip().lower() in keywords:
            toc_idx = i
            break

    start_idx = toc_idx + 1 if toc_idx != -1 else 0

    # 2. Quét để gom nhóm
    heading_con_sot_giua_2level = []
    prev_valid = None
    current_none_block = []

    for i in range(start_idx, len(struct_merged)):
        current_heading = struct_merged[i]

        if current_heading.get("level") is not None:
            if len(current_none_block) > 0:
                heading_con_sot_giua_2level.append({
                    "prev_valid": prev_valid,
                    "missing_headings": current_none_block,
                    "next_valid": current_heading
                })
                current_none_block = []
            prev_valid = current_heading
        else:
            current_none_block.append(current_heading)

    # Xử lý đoạn cuối file
    if len(current_none_block) > 0:
        heading_con_sot_giua_2level.append({
            "prev_valid": prev_valid,
            "missing_headings": current_none_block,
            "next_valid": None
        })

    return heading_con_sot_giua_2level

2.12 hàm tìm font_name

In [ ]:
import fitz

def enrich_font_name(struct_merged, pdf_path):
    """
    Quét lại file PDF dựa trên tọa độ (bbox) đã có trong struct_merged
    để trích xuất chính xác font_name và bổ sung vào dữ liệu.
    """
    pdf = fitz.open(pdf_path)

    for r in struct_merged:
        page = pdf[r["page"] - 1]
        page_height = page.rect.height

        # Tọa độ bbox trong struct_merged là gốc dưới-trái (chuẩn Docling)
        # Cần đổi sang chuẩn trên-trái của PyMuPDF để quét
        rect = fitz.Rect(
            r["bbox_l"],
            page_height - r["bbox_t"],
            r["bbox_r"],
            page_height - r["bbox_b"]
        )

        best_font = "Unknown"
        best_len = -1

        # Tìm cụm từ dài nhất trong vùng bbox này để lấy font chuẩn nhất
        for b in page.get_text("dict", clip=rect).get("blocks", []):
            for line in b.get("lines", []):
                for span in line.get("spans", []):
                    t = span["text"].strip()
                    if len(t) > best_len:
                        best_len = len(t)
                        best_font = span["font"]

        # Lưu vào dict
        r["font_name"] = best_font

    pdf.close()
    return struct_merged

2.13 xác định bậc cho các heading none còn lại trong struct_merged

In [ ]:
def assign_levels_for_missing_blocks(heading_con_sot_giua_2level):
    """
    Xác định và khôi phục level cho các khối heading bị None
    dựa trên font_name, font_size và khoảng cách tới heading trước.
    """
    for block in heading_con_sot_giua_2level:
        # 1. Tìm bậc nhỏ nhất 'a' (thứ hạng cao nhất, tức là giá trị số bé nhất)
        # giữa heading trước và sau khối này.
        levels = []
        if block["prev_valid"] and block["prev_valid"]["level"] is not None:
            levels.append(block["prev_valid"]["level"])
        if block["next_valid"] and block["next_valid"]["level"] is not None:
            levels.append(block["next_valid"]["level"])

        if levels:
            a = min(levels)
        else:
            a = 1  # Nếu khối lơ lửng không có cả trước lẫn sau

        # 2. Quét từ trên xuống dưới, phân nhóm theo font và gán bậc
        group_level_map = {}

        # Bậc của nhóm đầu tiên sẽ là a + 1 (Thấp hơn a 1 bậc)
        current_assign_level = a + 1

        for miss in block["missing_headings"]:
            f_name = miss.get("font_name", "Unknown")
            f_size = miss.get("font_size", 0)
            signature = (f_name, f_size)

            # Nếu gặp một định dạng font hoàn toàn mới trong khối này
            if signature not in group_level_map:
                group_level_map[signature] = current_assign_level
                current_assign_level += 1 # Nhóm tiếp theo sẽ bị tụt thêm 1 bậc

            # Gán level cho heading bị sót
            miss["level"] = group_level_map[signature]

2.14 hàm xóa những heading trong suspect -- là hàm kiếm các heading vị nhầm thành text -- bị nhầm thành text trong element_coords

In [ ]:
def filter_suspects_from_elements(suspect_rows, element_coords, tolerance=2.0):
    """
    So sánh tọa độ, in ra các phần tử bị nhầm thành suspect heading,
    đồng thời loại bỏ chúng ra khỏi danh sách và trả về element_coords thật sự.
    """
    real_element_coords = []
    removed_count = 0

    print(f"\nBắt đầu lọc suspect heading khỏi element_coords (sai số {tolerance}px)...")
    print(f"{'Page':<6} | {'Tọa độ (L, T, R, B)':<30} | {'Loại Element':<15} | {'Nội dung Text (Bị loại bỏ)'}")
    print("-" * 110)

    # Duyệt từng element, nếu trùng với bất kỳ suspect nào thì bỏ qua, nếu không thì giữ lại
    for elem in element_coords:
        e_page = elem.get('page')
        e_l = elem.get('bbox_l', 0)
        e_t = elem.get('bbox_t', 0)
        e_r = elem.get('bbox_r', 0)
        e_b = elem.get('bbox_b', 0)

        is_suspect = False

        for suspect in suspect_rows:
            s_page = suspect.get('page')
            if s_page != e_page:
                continue

            s_l, s_t, s_r, s_b = suspect.get('bbox', (0, 0, 0, 0))

            # Kiểm tra xem tọa độ có trùng không
            if (abs(s_l - e_l) <= tolerance and
                abs(s_t - e_t) <= tolerance and
                abs(s_r - e_r) <= tolerance and
                abs(s_b - e_b) <= tolerance):
                is_suspect = True
                break

        if is_suspect:
            # Nếu trùng -> in ra và không thêm vào real_element_coords
            removed_count += 1
            bbox_str = f"({e_l}, {e_t}, {e_r}, {e_b})"
            e_type = str(elem.get('type', ''))

            text_preview = str(elem.get('text', '')).replace('\n', ' ')
            if len(text_preview) > 45:
                text_preview = text_preview[:42] + "..."

            print(f"{e_page:<6} | {bbox_str:<30} | {e_type:<15} | {text_preview}")
        else:
            # Không trùng -> đây là element thật sự
            real_element_coords.append(elem)

    print("-" * 110)
    print(f"Đã loại bỏ {removed_count} suspect headings ra khỏi element_coords.\n")

    return real_element_coords

2.15 xác định text bị nhầm thành code -- xác định theo font monospace có sẵn trong element_coodrs rồi sau đó đi xét các đoạn , bảng và listitem

In [ ]:
def convert_monospace_text_to_code(element_coords, pdf_path, mono_threshold=0.8):
    _pdf = fitz.open(pdf_path)
    code_fonts = set()
    code_count = 0

    class _BBox:
        def __init__(self, l, t, r, b): self.l, self.t, self.r, self.b = l, t, r, b

    # ── Chỉ nhận font là monospace thật sự ───────────────────────────────────
    MONO_KEYWORDS = {
        "mono", "code", "courier", "consolas", "console",
        "typewriter", "fixed", "terminal", "hack", "menlo",
        "inconsolata", "iosevka", "fira", "jetbrains", "cascadia",
        "ubuntu mono", "source code", "roboto mono", "ibm plex mono",
        "lucida console", "anonymous", "droid sans mono",
    }

    def is_truly_monospace(font_name):
        f = font_name.lower()
        return any(kw in f for kw in MONO_KEYWORDS)

    # ── Học font code thật từ PDF (chỉ học nếu thật sự là mono) ─────────────
    for e in element_coords:
        if str(e.get("type")).lower() == "code":
            page = _pdf[e["page"] - 1]
            bbox = _BBox(e["bbox_l"], e["bbox_t"], e["bbox_r"], e["bbox_b"])
            info = dominant_span_info(page, bbox, page.rect.height)
            if info and is_truly_monospace(info["base_font"]):   # ← thêm điều kiện này
                code_fonts.add(info["base_font"].lower())
            code_count += 1
            if code_count >= 5: break

    default_monospace = {
        "ubuntumono", "ubuntu mono", "consolas",
        "courier", "courier new", "fira code", "jetbrains mono"
    }
    target_fonts = code_fonts | default_monospace

    # ── Tính tỉ lệ ký tự monospace trong 1 element ───────────────────────────
    def monospace_ratio(page, bbox, page_height):
        rect = fitz.Rect(bbox.l, page_height - bbox.t, bbox.r, page_height - bbox.b)
        total_chars = 0
        mono_chars  = 0
        for b in page.get_text("dict", clip=rect)["blocks"]:
            for line in b.get("lines", []):
                for span in line.get("spans", []):
                    t = span["text"].strip()
                    if not t:
                        continue
                    char_count   = len(t)
                    total_chars += char_count
                    bfont = base_font(span["font"]).lower()
                    if any(f in bfont for f in target_fonts):
                        mono_chars += char_count
        if total_chars == 0:
            return 0.0
        return mono_chars / total_chars

    # ── Duyệt và chuyển đổi ──────────────────────────────────────────────────
    converted_count = 0
    for e in element_coords:
        original_type = str(e.get("type")).lower()
        if original_type not in ["text", "list_item", "table"]:
            continue
        page  = _pdf[e["page"] - 1]
        bbox  = _BBox(e["bbox_l"], e["bbox_t"], e["bbox_r"], e["bbox_b"])
        ratio = monospace_ratio(page, bbox, page.rect.height)
        if ratio >= mono_threshold:
            e["original_type"] = original_type
            e["type"]          = "code"
            converted_count   += 1

    _pdf.close()
    print(f" Đã chuyển đổi thành công {converted_count} đoạn thành code!")
    return element_coords

2.16 đoạn code bị nhầm trong hình ảnh cắt hình ảnh thành code và ảnh

In [ ]:
import fitz
import re

def split_picture_containing_code(element_coords, pdf_path):
    _pdf = fitz.open(pdf_path)
    def get_base_font(fname):
        return re.sub(r"[-,]?(bold|italic|regular|roman|light|medium|black|heavy|demi|semibold|oblique).*", "", fname, flags=re.IGNORECASE).strip().lower()

    code_fonts = set()
    code_count = 0
    class _BBox:
        def __init__(self, l, t, r, b): self.l, self.t, self.r, self.b = l, t, r, b

    for e in element_coords:
        if str(e.get("type")).lower() == "code":
            page = _pdf[e["page"] - 1]
            bbox = _BBox(e["bbox_l"], e["bbox_t"], e["bbox_r"], e["bbox_b"])
            info = dominant_span_info(page, bbox, page.rect.height)
            if info: code_fonts.add(info["base_font"].lower())
            code_count += 1
            if code_count >= 5: break

    default_monospace = {
        # Cổ điển & Mặc định hệ thống
        "courier", "courier new", "consolas", "monaco", "lucida console",
        "andale mono", "fixedsys", "terminal", "vga", "freemono", "nimbus mono",
        "courier prime", "consolemono", "prestige elite", "letter gothic",
        "system mono", "monospace", "lucida sans typewriter", "ms gothic",
        "ms mincho", "couriernewpsmt", "courier-bold", "courier-oblique",

        # Các font lập trình hiện đại & Nổi tiếng nhất
        "fira code", "firacode", "fira mono", "source code pro", "sourcecode",
        "jetbrains mono", "jetbrainsmono", "cascadia code", "cascadiacode", "cascadia mono",
        "ubuntu mono", "ubuntumono", "roboto mono", "robotomono", "hack", "menlo",
        "inconsolata", "sf mono", "sfmono", "ibm plex mono", "ibmplexmono",
        "sf mono compact", "menlo regular", "cascadia code pl", "cascadia code nf",

        # Họ font mã nguồn mở & Linux
        "dejavu sans mono", "dejavumono", "bitstream vera sans mono", "liberation mono",
        "droid sans mono", "noto sans mono", "oxygen mono", "pt mono", "space mono",
        "spacemono", "dejavusansmono",

        # Các font lập trình chuyên dụng (Custom/Hacker)
        "anonymous pro", "iosevka", "victor mono", "operator mono", "dank mono",
        "pragmatapro", "meslo", "terminus", "monoid", "go mono", "input mono",
        "envy code r", "fantasque sans mono", "monofur", "cutive mono", "share tech mono",
        "nova mono", "vt323", "syne mono", "xanh mono", "b612 mono", "cousine",
        "overpass mono", "spleen", "agave", "ocr a", "ocr-a", "proggy", "m+ mono",
        "comic mono",

        # Font hiện đại khác
        "recursive mono", "commit mono", "commitmono", "red hat mono",
        "geist mono", "berkeley mono", "maple mono", "sarasa mono",
        "0xproto", "martian mono",

        # Font Nerd Font
        "firacode nerd font", "jetbrainsmono nerd font", "hack nerd font",
        "meslo nerd font", "cascadia nerd font",

        # Font monospace tiếng Việt/CJK hay gặp
        "noto sans mono cjk", "sarasa term", "sarasa fixed",

        # Font cổ/hiếm
        "prestige", "line printer", "teletype", "px437", "perfect dos vga 437",

        # Biến thể viết liền không dấu cách
        "couriernew", "lucidaconsole", "sourcecodepro", "ibmplexmono",
        "robotomono", "firamono",

        # Font hay gặp trong file PDF
        "latin modern mono", "latinmodernmono", "computer modern typewriter",
        "computer modern mono", "cmtt", "cmtt10", "lmmono", "lmmono10",
        "nimbus mono ps", "courier10bt", "prestige elite std",
    }

    target_fonts = code_fonts | default_monospace

    new_element_coords = []
    split_count = 0

    for e in element_coords:
        if str(e.get("type")).lower() == "picture":
            page = _pdf[e["page"] - 1]
            page_height = page.rect.height
            pic_l, pic_r = e["bbox_l"], e["bbox_r"]

            y0 = page_height - e["bbox_t"]
            y1 = page_height - e["bbox_b"]
            rect = fitz.Rect(pic_l, y0, pic_r, y1)

            mono_spans = []
            blocks = page.get_text("dict", clip=rect).get("blocks", [])
            for b in blocks:
                for line in b.get("lines", []):
                    for span in line.get("spans", []):
                        text = span.get("text", "").strip()
                        if not text: continue
                        bfont = get_base_font(span["font"])
                        if any(f in bfont for f in target_fonts):
                            mono_spans.append(span["bbox"])

            if mono_spans:
                mono_y0 = min(s[1] for s in mono_spans)
                mono_y1 = max(s[3] for s in mono_spans)
                dist_to_top = abs(mono_y0 - y0)
                dist_to_bottom = abs(y1 - mono_y1)

                code_elem = e.copy()
                code_elem["type"] = "code"

                if dist_to_top <= dist_to_bottom:
                    code_y0, code_y1 = y0, mono_y1
                    pic_new_y0, pic_new_y1 = mono_y1 + 5, y1
                    code_goes_first = True
                else:
                    code_y0, code_y1 = mono_y0, y1
                    pic_new_y0, pic_new_y1 = y0, mono_y0 - 5
                    code_goes_first = False

                if pic_new_y1 > pic_new_y0:
                    code_elem["bbox_t"] = round(page_height - code_y0, 1)
                    code_elem["bbox_b"] = round(page_height - code_y1, 1)
                    code_rect = fitz.Rect(pic_l, code_y0, pic_r, code_y1)
                    code_elem["text"] = page.get_text("text", clip=code_rect).strip()
                    # THÊM NHÃN GỐC Ở ĐÂY
                    code_elem["original_type"] = "picture_split"

                    e["bbox_t"] = round(page_height - pic_new_y0, 1)
                    e["bbox_b"] = round(page_height - pic_new_y1, 1)

                    if code_goes_first: new_element_coords.extend([code_elem, e])
                    else: new_element_coords.extend([e, code_elem])
                    split_count += 1
                    continue
                else:
                    e["type"] = "code"
                    code_rect = fitz.Rect(pic_l, code_y0, pic_r, code_y1)
                    e["text"] = page.get_text("text", clip=code_rect).strip()
                    # THÊM NHÃN GỐC Ở ĐÂY
                    e["original_type"] = "picture_full"
                    new_element_coords.append(e)
                    continue

        new_element_coords.append(e)

    _pdf.close()
    return new_element_coords

2.17 Xác định lại những hình ảnh bị nhầm thành table

In [ ]:
def fix_tables_misidentified_as_pictures(element_coords, doc, caption_keywords=("hình", "figure")):
    """
    Duyệt các TABLE trong doc, nếu caption đi kèm BẮT ĐẦU BẰNG "hình"/"figure"
    thì coi đây thực chất là PICTURE bị nhận nhầm thành TABLE -> đổi type
    tương ứng trong element_coords (dùng bbox từ find_heading để định vị
    đúng entry cần sửa).

    Trả về: element_coords đã sửa (list mới), danh sách các entry đã đổi type.
    """
    from docling_core.types.doc import DocItemLabel

    def _get_caption_text_for_item(item):
        captions = getattr(item, "captions", None) or []
        texts = []
        for cap_ref in captions:
            try:
                cap_item = cap_ref.resolve(doc)
            except AttributeError:
                cap_item = cap_ref
            if cap_item is not None:
                texts.append(getattr(cap_item, "text", "") or "")
        return " ".join(texts)

    def _bbox_match(e, prov_bbox, tol=1.0):
        return (
            abs(e["bbox_l"] - prov_bbox.l) <= tol and
            abs(e["bbox_t"] - prov_bbox.t) <= tol and
            abs(e["bbox_r"] - prov_bbox.r) <= tol and
            abs(e["bbox_b"] - prov_bbox.b) <= tol
        )

    new_element_coords = [dict(e) for e in element_coords]
    fixed_entries = []

    for item, _level in doc.iterate_items():
        if item.label != DocItemLabel.TABLE or not item.prov:
            continue

        caption_text = _get_caption_text_for_item(item).strip().lower()
        is_actually_picture = any(caption_text.startswith(kw) for kw in caption_keywords)
        if not is_actually_picture:
            continue

        prov = item.prov[0]
        for e in new_element_coords:
            if e["page"] == prov.page_no and e["type"] == "table" and _bbox_match(e, prov.bbox):
                e["type"] = "picture"
                fixed_entries.append({
                    "page": e["page"],
                    "bbox": (e["bbox_l"], e["bbox_t"], e["bbox_r"], e["bbox_b"]),
                    "old_type": "table",
                    "new_type": "picture",
                    "caption": caption_text[:80],
                })

    print(f" Đã sửa {len(fixed_entries)} TABLE nhận nhầm -> PICTURE (theo caption 'hình'/'figure')")
    for f in fixed_entries:
        print(f"  trang {f['page']:<4} bbox={f['bbox']}  caption='{f['caption']}'")

    return new_element_coords, fixed_entries

2.18 gom các hình bị nhận diện nhầm là tách nhau -- nhưng vẫn có lỗi bởi vì model detect có khi detect thiếu caption

In [ ]:
def merge_pictures_by_coordinates(pdf_path, element_coords, struct_merged, doc, edge_tolerance=20.0):
    """
    Gom nhóm các PICTURE trên cùng 1 trang dựa theo toạ độ:
      - Cùng trang
      - Có ít nhất 1 mốc (trên/dưới/trái/phải) lệch nhau < edge_tolerance
      - Không có vật cản (table/text/heading/ảnh khác/caption) chen giữa
        theo đúng phương đang xét
    Gộp dây chuyền (union-find) để xử lý được nhóm 3-4 ảnh liên tiếp.
    Ảnh sau khi gộp lấy caption/text từ ảnh NÀO TRONG NHÓM CÓ CAPTION
    (tra qua doc.iterate_items() + item.captions); nếu cả nhóm không ảnh
    nào có caption thì lấy field từ ảnh đầu tiên như cũ.

    Trả về: element_coords mới (đã gộp), danh sách các nhóm đã gộp.
    """
    from docling_core.types.doc import DocItemLabel

    def _get_all_obstacles_on_page(page_no):
        obstacles = []
        for e in element_coords:
            if e["page"] != page_no or e["type"] == "picture":
                continue
            obstacles.append((e["bbox_l"], e["bbox_t"], e["bbox_r"], e["bbox_b"], e["type"]))
        for r in struct_merged:
            if r["page"] != page_no:
                continue
            obstacles.append((r["bbox_l"], r["bbox_t"], r["bbox_r"], r["bbox_b"], "heading"))
        for item, _level in doc.iterate_items():
            if item.label != DocItemLabel.CAPTION or not item.prov:
                continue
            prov = item.prov[0]
            if prov.page_no != page_no:
                continue
            bbox = prov.bbox
            obstacles.append((bbox.l, bbox.t, bbox.r, bbox.b, "caption"))
        return obstacles

    def _is_between_horizontal(obstacle, box_a, box_b):
        o_l, o_t, o_r, o_b, _kind = obstacle
        l_a, t_a, r_a, b_a = box_a
        l_b, t_b, r_b, b_b = box_b

        left_box, right_box = (box_a, box_b) if l_a < l_b else (box_b, box_a)
        gap_l = left_box[2]
        gap_r = right_box[0]
        if gap_l >= gap_r:
            return False

        if o_r < gap_l or o_l > gap_r:
            return False

        min_t = min(t_a, t_b)
        max_b = max(b_a, b_b)
        if o_b > min_t or o_t < max_b:
            return False

        return True

    def _is_between_vertical(obstacle, box_a, box_b):
        o_l, o_t, o_r, o_b, _kind = obstacle
        l_a, t_a, r_a, b_a = box_a
        l_b, t_b, r_b, b_b = box_b

        top_box, bottom_box = (box_a, box_b) if t_a > t_b else (box_b, box_a)
        gap_t = top_box[3]
        gap_b = bottom_box[1]
        if gap_b >= gap_t:
            return False

        if o_t < gap_b or o_b > gap_t:
            return False

        min_l = min(l_a, l_b)
        max_r = max(r_a, r_b)
        if o_r < min_l or o_l > max_r:
            return False

        return True

    def _find_shared_edge_direction(box_a, box_b, tol):
        l1, t1, r1, b1 = box_a
        l2, t2, r2, b2 = box_b

        same_top    = abs(t1 - t2) <= tol
        same_bottom = abs(b1 - b2) <= tol
        same_left   = abs(l1 - l2) <= tol
        same_right  = abs(r1 - r2) <= tol

        if same_top or same_bottom:
            return "horizontal"
        if same_left or same_right:
            return "vertical"
        return None

    def _can_merge_pair(box_a, box_b, page_no):
        direction = _find_shared_edge_direction(box_a, box_b, edge_tolerance)
        if direction is None:
            return False

        obstacles = _get_all_obstacles_on_page(page_no)
        check_fn = _is_between_horizontal if direction == "horizontal" else _is_between_vertical

        for obs in obstacles:
            if check_fn(obs, box_a, box_b):
                return False

        return True

    def _bbox_match(e, prov_bbox, tol=1.0):
        return (
            abs(e["bbox_l"] - prov_bbox.l) <= tol and
            abs(e["bbox_t"] - prov_bbox.t) <= tol and
            abs(e["bbox_r"] - prov_bbox.r) <= tol and
            abs(e["bbox_b"] - prov_bbox.b) <= tol
        )

    def _get_caption_text_for_entry(entry):
        """Tra trong doc xem entry (1 picture trong element_coords) có caption
        thật sự đi kèm không (qua item.captions). Trả về text caption hoặc None."""
        for item, _level in doc.iterate_items():
            if item.label != DocItemLabel.PICTURE or not item.prov:
                continue
            prov = item.prov[0]
            if prov.page_no != entry["page"] or not _bbox_match(entry, prov.bbox):
                continue
            captions = getattr(item, "captions", None) or []
            for cap_ref in captions:
                try:
                    cap_item = cap_ref.resolve(doc)
                except AttributeError:
                    cap_item = cap_ref
                if cap_item is not None and getattr(cap_item, "text", ""):
                    return cap_item.text
        return None

    pictures = [
        (idx, e) for idx, e in enumerate(element_coords) if e["type"] == "picture"
    ]

    parent = {idx: idx for idx, _ in pictures}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry

    by_page = {}
    for idx, e in pictures:
        by_page.setdefault(e["page"], []).append((idx, e))

    for page_no, items in by_page.items():
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                idx_a, e_a = items[i]
                idx_b, e_b = items[j]
                box_a = (e_a["bbox_l"], e_a["bbox_t"], e_a["bbox_r"], e_a["bbox_b"])
                box_b = (e_b["bbox_l"], e_b["bbox_t"], e_b["bbox_r"], e_b["bbox_b"])
                if _can_merge_pair(box_a, box_b, page_no):
                    union(idx_a, idx_b)

    groups = {}
    for idx, _e in pictures:
        root = find(idx)
        groups.setdefault(root, []).append(idx)

    new_element_coords = [dict(e) for e in element_coords]
    merged_groups_info = []
    indices_to_remove = set()

    for root, idx_list in groups.items():
        if len(idx_list) < 2:
            continue

        boxes = [
            (
                new_element_coords[i]["bbox_l"],
                new_element_coords[i]["bbox_t"],
                new_element_coords[i]["bbox_r"],
                new_element_coords[i]["bbox_b"],
            )
            for i in idx_list
        ]
        ls = [b[0] for b in boxes]
        ts = [b[1] for b in boxes]
        rs = [b[2] for b in boxes]
        bs = [b[3] for b in boxes]
        merged_bbox = (min(ls), max(ts), max(rs), min(bs))

        # ── Tìm ảnh nào trong nhóm CÓ caption thật sự, lấy làm base_entry ──
        base_idx = idx_list[0]
        caption_text_found = None
        for i in idx_list:
            cap_text = _get_caption_text_for_entry(new_element_coords[i])
            if cap_text:
                base_idx = i
                caption_text_found = cap_text
                break

        base_entry = dict(new_element_coords[base_idx])
        base_entry.update({
            "bbox_l": round(merged_bbox[0], 1),
            "bbox_t": round(merged_bbox[1], 1),
            "bbox_r": round(merged_bbox[2], 1),
            "bbox_b": round(merged_bbox[3], 1),
        })
        if caption_text_found:
            base_entry["text"] = caption_text_found[:120]

        for i in idx_list:
            indices_to_remove.add(i)

        new_element_coords.append(base_entry)
        merged_groups_info.append({
            "page": base_entry["page"],
            "count_merged": len(idx_list),
            "merged_bbox": merged_bbox,
            "caption_used": caption_text_found[:80] if caption_text_found else None,
        })

    final_element_coords = [
        e for i, e in enumerate(new_element_coords) if i not in indices_to_remove
    ]

    print(f" Đã gộp {len(merged_groups_info)} nhóm hình theo toạ độ")
    for g in merged_groups_info:
        cap_info = f"caption='{g['caption_used']}'" if g['caption_used'] else "(không có caption)"
        print(f"  trang {g['page']:<4} gộp {g['count_merged']} hình -> bbox={g['merged_bbox']}  {cap_info}")

    return final_element_coords, merged_groups_info

2.19 thêm 1 cột caption vào trong element_coords để đưa caption vào trong hình ảnh và table

In [ ]:
def add_caption_field_to_element_coords(element_coords, doc):
    """
    Thêm field "caption" (mặc định None) vào MỌI entry trong element_coords.
    Chỉ những entry PICTURE/TABLE nào thực sự có caption (tra qua item.captions
    trong doc) mới được điền giá trị caption tương ứng; các entry khác giữ None.
    """
    from docling_core.types.doc import DocItemLabel

    def _bbox_match(e, prov_bbox, tol=1.0):
        return (
            abs(e["bbox_l"] - prov_bbox.l) <= tol and
            abs(e["bbox_t"] - prov_bbox.t) <= tol and
            abs(e["bbox_r"] - prov_bbox.r) <= tol and
            abs(e["bbox_b"] - prov_bbox.b) <= tol
        )

    new_element_coords = [dict(e) for e in element_coords]

    # Mặc định caption = None cho toàn bộ entry
    for e in new_element_coords:
        e["caption"] = None

    # Chỉ tra caption cho PICTURE / TABLE (khớp bbox với doc)
    for item, _level in doc.iterate_items():
        if item.label not in (DocItemLabel.PICTURE, DocItemLabel.TABLE) or not item.prov:
            continue

        captions = getattr(item, "captions", None) or []
        caption_texts = []
        for cap_ref in captions:
            try:
                cap_item = cap_ref.resolve(doc)
            except AttributeError:
                cap_item = cap_ref
            if cap_item is not None and getattr(cap_item, "text", ""):
                caption_texts.append(cap_item.text)

        if not caption_texts:
            continue

        prov = item.prov[0]
        for e in new_element_coords:
            if e["page"] == prov.page_no and e["type"] == item.label.value and _bbox_match(e, prov.bbox):
                e["caption"] = " ".join(caption_texts)

    n_with_caption = sum(1 for e in new_element_coords if e["caption"] is not None)
    print(f" Đã thêm field 'caption' cho {len(new_element_coords)} entry, trong đó {n_with_caption} entry có caption")

    return new_element_coords

cell luồng chạy xử lý cho ra heading đươc

In [ ]:
import fitz
from docling_core.types.doc import DocItemLabel

def find_heading(pdf_path, doc):

    heading_raw_elements, heading_signatures = get_bold_headings(doc, pdf_path)

    # ── BƯỚC 1.4: Lấy heading từ Docling ─────────────────────────────────────
    struct_raw, rows = build_struct_raw(doc, pdf_path)

    # ── BƯỚC 1.6: Tìm heading bị Docling nhận nhầm + thu thập tọa độ elements ─
    def bold_not_preceded_by_normal(page, bbox, page_height) -> bool:
        rect = fitz.Rect(bbox.l, page_height - bbox.t, bbox.r, page_height - bbox.b)
        blocks = page.get_text("dict", clip=rect)["blocks"]
        for b in blocks:
            for line in b.get("lines", []):
                spans = [s for s in line.get("spans", []) if s["text"].strip()]
                found_bold = False
                for span in spans:
                    if is_bold(span):
                        found_bold = True
                    else:
                        if not found_bold:
                            return False
        return True

    _SUSPECT_LABELS = {DocItemLabel.TEXT, DocItemLabel.LIST_ITEM}
    _pdf = fitz.open(pdf_path)
    suspect_rows   = []
    element_coords = []

    for item, _level in doc.iterate_items():
        if item.label in (
            DocItemLabel.TEXT, DocItemLabel.FORMULA, DocItemLabel.PICTURE,
            DocItemLabel.TABLE, DocItemLabel.LIST_ITEM, DocItemLabel.CODE,
        ) and item.prov:
            prov = item.prov[0]
            bbox = prov.bbox
            # Thành thế này:
            page_obj = _pdf[prov.page_no - 1]
            info = dominant_span_info(page_obj, bbox, page_obj.rect.height)   # ← đọc font ngay tại đây

            element_coords.append({
                "type"     : item.label.value,
                "page"     : prov.page_no,
                "bbox_l"   : round(bbox.l, 1),
                "bbox_t"   : round(bbox.t, 1),
                "bbox_r"   : round(bbox.r, 1),
                "bbox_b"   : round(bbox.b, 1),
                "text"     : getattr(item, "text", "")[:120],
                "font_name": info["font"] if info else "unknown",       # ← lưu luôn
                "base_font": info["base_font"] if info else "unknown",  # ← lưu luôn (tên sạch)
                "font_size": info["size"] if info else 0,               # ← lưu luôn
            })

        if item.label not in _SUSPECT_LABELS or not item.prov: continue
        raw_text = getattr(item, "text", "")
        if "\n" in raw_text or len(raw_text) > 200: continue
        prov = item.prov[0]
        page = _pdf[prov.page_no - 1]
        bbox = prov.bbox

        rect = fitz.Rect(bbox.l, page.rect.height - bbox.t, bbox.r, page.rect.height - bbox.b)
        blocks = page.get_text("dict", clip=rect)["blocks"]
        if sum(len(b.get("lines", [])) for b in blocks) != 1: continue

        info = dominant_span_info(page, bbox, page.rect.height)
        if info is None or not info["bold"]: continue
        if (info["base_font"], info["size"]) not in heading_signatures: continue
        if not bold_not_preceded_by_normal(page, bbox, page.rect.height): continue

        suspect_rows.append({
            "label": item.label.value, "page": prov.page_no, "font": info["font"],
            "base_font": info["base_font"], "size": info["size"], "text": raw_text[:80],
            "bbox": (round(bbox.l,1), round(bbox.t,1), round(bbox.r,1), round(bbox.b,1)),
        })

    _pdf.close()

    # Ghi đè lại element_coords bằng danh sách thật sự (đã loại bỏ suspect)
    element_coords = filter_suspects_from_elements(suspect_rows, element_coords, tolerance=2.0)

    # HÀM CHUYỂN TEXT -> CODE
    element_coords = convert_monospace_text_to_code(element_coords, pdf_path)

    # Cắt các bức ảnh bị dính Code làm đôi
    element_coords = split_picture_containing_code(element_coords, pdf_path)


    # ── BƯỚC 1.7: Gộp suspect vào rows ───────────────────────────────────────
    size_to_level = {r["font_size"]: r["level"] for r in rows}
    normalized = []
    for r in rows:
        normalized.append({
            "level": r["level"], "text": r["text"], "page": r["page"],
            "font_size": r["font_size"], "bbox_l": r["bbox_l"], "bbox_t": r["bbox_t"],
            "bbox_r": r["bbox_r"], "bbox_b": r["bbox_b"], "source": "docling",
        })

    for r in suspect_rows:
        level = size_to_level.get(r["size"])
        if level is None:
            known = sorted(size_to_level.items(), key=lambda x: x[0])
            level = 1
            for sz, lv in known:
                if r["size"] >= sz: level = lv
        l, t, r_, b = r["bbox"]
        normalized.append({
            "level": level, "text": r["text"], "page": r["page"], "font_size": r["size"],
            "bbox_l": l, "bbox_t": t, "bbox_r": r_, "bbox_b": b, "source": "suspect",
        })

    seen, deduped = set(), []
    for r in normalized:
        key = (r["page"], r["text"].strip().lower()[:60])
        if key not in seen:
            seen.add(key)
            deduped.append(r)

    struct_merged = sorted(deduped, key=lambda r: (r["page"], -r["bbox_t"]))

    # Thêm font_name cho tất cả các heading
    struct_merged = enrich_font_name(struct_merged, pdf_path)

    struct_merged, removed_headings = filter_monospace_headings(struct_merged, element_coords, pdf_path)

    element_coords, removed_headings_log = resolve_removed_headings(
        struct_merged, removed_headings, element_coords
    )

    # Sửa các TABLE bị nhận nhầm từ PICTURE (dựa vào caption "hình"/"figure")
    element_coords, fixed_table_to_picture = fix_tables_misidentified_as_pictures(element_coords, doc)

    # Gộp các picture bị tách rời theo toạ độ (cùng mốc, không vật cản chen giữa)
    element_coords, merged_pictures_info = merge_pictures_by_coordinates(
        pdf_path, element_coords, struct_merged, doc
    )

    # Thêm field "caption" (None mặc định) cho mọi entry trong element_coords
    element_coords = add_caption_field_to_element_coords(element_coords, doc)

    # ── BƯỚC 2.2 & 2.3: Chạy luồng Mục Lục (TOC) ────────────────────────────
    real_heading_muc_luc = extract_toc_headings(pdf_path)

    if real_heading_muc_luc:
        idx_toc, no_idx_toc, grouped_heading = process_toc_v2(real_heading_muc_luc)
        toc_heading_level, toc_tree, _ = assign_levels_and_build_tree(grouped_heading, real_heading_muc_luc, struct_merged)

        toc_items_list = []
        for sig, items in grouped_heading.items():
            toc_items_list.extend(items)
        toc_items_list.sort(key=lambda x: x.get("original_order", 0))
    else:
        toc_heading_level, toc_tree = [], ""
        toc_items_list = []

    bookmark_tree = extract_bookmark_tree(pdf_path)

    if bookmark_tree:
        import difflib
        real_heading_muc_luc_level = []
        last_matched_idx = 0

        for item in bookmark_tree:
            norm_title = "".join(str(item["title"]).split()).lower()
            best_idx, best_ratio, best_k = "", 0.0, last_matched_idx
            window_end = min(last_matched_idx + 15, len(toc_items_list))

            for k in range(last_matched_idx, window_end):
                toc_item = toc_items_list[k]
                norm_toc_text = "".join(str(toc_item.get("text", "")).split()).lower()
                ratio = difflib.SequenceMatcher(None, norm_title, norm_toc_text).ratio()
                if ratio > best_ratio:
                    best_ratio, best_idx, best_k = ratio, toc_item.get("extracted_index", ""), k

            if best_ratio >= 0.5:
                idx = best_idx
                last_matched_idx = best_k + 1
            else:
                idx = ""

            real_heading_muc_luc_level.append({
                "text": item["title"], "level": item["level"],
                "page_num": item["page"], "extracted_index": idx,
            })

        tree_lines = ["    " * (item["level"] - 1) + "- " + item["title"] for item in bookmark_tree]
        heading_muc_luc_tree = "\n".join(tree_lines)
    else:
        real_heading_muc_luc_level = toc_heading_level
        heading_muc_luc_tree = toc_tree



    # ── PHẦN 3: ĐỒNG BỘ XUỐNG VĂN BẢN ───────────────────────────────────────
    struct_merged = assign_levels_to_struct_merged(struct_merged, real_heading_muc_luc_level)

    heading_con_sot_giua_2level = get_missing_level_blocks(struct_merged)

    # Gán level tự động cho các khối bị sót
    assign_levels_for_missing_blocks(heading_con_sot_giua_2level)

    # ── BÁO CÁO NHANH GỌN LẸ ────────────────────────────────────────────────
    print(f" ĐÃ XỬ LÝ XONG: Tìm thấy {len(struct_merged)} heading và lấp đầy {len(heading_con_sot_giua_2level)} khối bị sót.")

    return struct_merged, real_heading_muc_luc_level, heading_muc_luc_tree, element_coords, heading_con_sot_giua_2level, removed_headings_log

3.1 Hàm gán và thêm các trường trong chunk do AI tự biên tự diễn

In [ ]:
import fitz
import os

# MODULE MỚI: Lọc bỏ phần Mục Lục (Chỉ lấy nội dung thật)
def filter_elements_after_toc(element_coords, struct_merged):
    # 1. Tìm heading Mục lục
    toc_heading_idx = -1
    for i, h in enumerate(struct_merged):
        text_lower = h.get("text", "").strip().lower()
        if "mục lục" in text_lower or "table of contents" in text_lower:
            toc_heading_idx = i
            break

    if toc_heading_idx == -1:
        print(" Không tìm thấy Heading 'Mục lục'. Giữ nguyên toàn bộ dữ liệu.")
        return element_coords

    toc_level = struct_merged[toc_heading_idx].get("level")
    if toc_level is None:
        toc_level = 99

    # 2. Tìm Heading nội dung đầu tiên ngay SAU Mục lục
    # (Là heading có level nhỏ hơn hoặc bằng level của Mục lục)
    start_content_idx = -1
    for i in range(toc_heading_idx + 1, len(struct_merged)):
        lvl = struct_merged[i].get("level")
        if lvl is None:
            lvl = 99
        if lvl <= toc_level:
            start_content_idx = i
            break

    if start_content_idx == -1:
        print(" Không tìm thấy nội dung sau Mục lục. Trả về toàn bộ dữ liệu.")
        return element_coords

    start_heading = struct_merged[start_content_idx]
    start_page = start_heading["page"]
    start_y = start_heading["bbox_t"]

    # 3. Lọc bỏ các element đứng TRƯỚC start_heading
    filtered_coords = []
    for elem in element_coords:
        # Nằm sau nếu: Trang lớn hơn HOẶC (Cùng trang và Tọa độ Y nhỏ hơn/bằng)
        # (Y càng nhỏ tức là càng nằm dưới cùng của trang)
        if elem["page"] > start_page or (elem["page"] == start_page and elem["bbox_t"] <= start_y):
            filtered_coords.append(elem)

    bi_loai_bo = len(element_coords) - len(filtered_coords)
    print(f" Đã cắt bỏ phần Mục Lục. Xóa đi {bi_loai_bo} elements rác. Giữ lại {len(filtered_coords)} elements nội dung.")

    return filtered_coords

# MODULE 1: Gắn Gia Phả (Heading Path) cho từng Element
def assign_heading_paths(element_coords, struct_merged):
    # Bước 1: Tiền xử lý struct_merged để tạo đường dẫn (path) cho mỗi heading
    path_stack = []

    for i, heading in enumerate(struct_merged):
        level = heading.get("level")
        if level is None:
            level = 99 # Xử lý fallback nếu sót level

        # Xóa các heading trong stack có level >= level hiện tại
        path_stack = [h for h in path_stack if h["level"] < level]

        # Heading cha chính là phần tử cuối cùng còn lại trong stack
        parent_text = path_stack[-1]["text"] if path_stack else "Root"

        # Đưa heading hiện tại vào stack
        path_stack.append({"level": level, "text": heading["text"]})

        # Tạo chuỗi heading_path
        heading_path = " > ".join([h["text"] for h in path_stack])

        # Tìm children (các heading phía dưới có level = level + 1)
        children = []
        for j in range(i + 1, len(struct_merged)):
            next_level = struct_merged[j].get("level")
            if next_level is None:
                next_level = 99
            if next_level <= level:
                break # Gặp heading đồng cấp hoặc to hơn thì dừng
            if next_level == level + 1:
                children.append(struct_merged[j]["text"])

        # Lưu lại thông tin vào struct_merged
        heading["heading_path"] = heading_path
        heading["heading_parent"] = parent_text
        heading["heading_children"] = children

    # Bước 2: Gắn heading path vào từng element trong element_coords
    for elem in element_coords:
        nearest_heading = None

        # Dò ngược struct_merged để tìm heading nằm ngay trên element này
        for heading in struct_merged:
            # Điều kiện nằm trên: Trang nhỏ hơn HOẶC (Cùng trang và tọa độ Y lớn hơn/bằng)
            # Lưu ý: Docling lấy gốc tọa độ Y ở dưới cùng trang, nên Y càng lớn tức là càng nằm bên trên
            if heading["page"] < elem["page"] or (heading["page"] == elem["page"] and heading["bbox_t"] >= elem["bbox_t"]):
                if nearest_heading is None or heading["page"] > nearest_heading["page"] or (heading["page"] == nearest_heading["page"] and heading["bbox_t"] <= nearest_heading["bbox_t"]):
                    nearest_heading = heading

        if nearest_heading:
            elem["heading_path"] = nearest_heading["text"] # Lấy đúng TÊN của Heading hiện tại
            elem["heading_parent"] = nearest_heading["heading_parent"]
            elem["heading_children"] = nearest_heading["heading_children"]
        else:
            elem["heading_path"] = "Root"
            elem["heading_parent"] = "Root"
            elem["heading_children"] = []

    print(" Đã gắn xong Gia Phả (Heading Path) cho các elements.")
    return element_coords

# MODULE 2: Gắn Ngữ Cảnh Xung Quanh (Context Snippets Sát Rạt)
def assign_context_snippets(element_coords, max_chars=300):
    for i, elem in enumerate(element_coords):
        if elem["type"] == "text":
            continue

        prev_text = ""
        next_text = ""

        # Liền trước: Phải là text VÀ không bị chắn bởi Heading (cùng chung heading_path)
        if i > 0:
            prev_elem = element_coords[i-1]
            if prev_elem["type"] == "text" and prev_elem.get("heading_path") == elem.get("heading_path"):
                prev_text = prev_elem.get("text", "")

        # Liền sau: Phải là text VÀ không bị chắn bởi Heading
        if i < len(element_coords) - 1:
            next_elem = element_coords[i+1]
            if next_elem["type"] == "text" and next_elem.get("heading_path") == elem.get("heading_path"):
                next_text = next_elem.get("text", "")

        elem["prev_text_snippet"] = prev_text[:max_chars]
        elem["next_text_snippet"] = next_text[:max_chars]

    print(" Đã gắn xong Context Snippets (Chỉ lấy text sát rạt, không bị chắn).")
    return element_coords

# MODULE 2.5: Trích xuất nội dung Bảng theo chiều ngang
def extract_full_table_text(element_coords, doc):
    from docling_core.types.doc import DocItemLabel

    for item, _level in doc.iterate_items():
        if item.label == DocItemLabel.TABLE and item.prov:
            prov = item.prov[0]
            # Dò tìm element table tương ứng trong element_coords
            for elem in element_coords:
                if elem["type"] == "table" and elem["page"] == prov.page_no:
                    # Sai số tọa độ nhỏ hơn 2.0
                    if abs(elem["bbox_t"] - prov.bbox.t) < 2.0:
                        try:
                            df = item.export_to_dataframe(doc)
                            # Ghép các ô thành hàng ngang, cách nhau bởi dấu "|"
                            horizontal_lines = []
                            for row_idx, row in enumerate(df.values):
                                line = " | ".join([str(val).replace('\n', ' ') if val is not None else "" for val in row])
                                horizontal_lines.append(f"Hàng {row_idx + 1}: {line}")

                            elem["table_horizontal_text"] = "\n".join(horizontal_lines)
                        except Exception as e:
                            elem["table_horizontal_text"] = f"Lỗi đọc bảng: {str(e)}"

    print(" Đã quét và đọc nội dung Table theo chiều ngang liền mạch.")
    return element_coords

# MODULE 3: Cắt Hình Ảnh Từ PDF
def crop_and_upload_images(element_coords, local_pdf_path, output_dir="cropped_images", supabase_client=None, doc_ai_id="unknown_doc"):
    """
    Cắt ảnh từ PDF, lưu tạm ra ổ cứng, sau đó upload lên Supabase Storage và lấy link public.
    """
    os.makedirs(output_dir, exist_ok=True)
    pdf = fitz.open(local_pdf_path)

    count = 0
    for i, elem in enumerate(element_coords):
        if elem["type"] == "picture":
            page = pdf[elem["page"] - 1]
            page_height = page.rect.height

            # Chuyển đổi hệ tọa độ: Docling (gốc dưới-trái) sang PyMuPDF (gốc trên-trái)
            y0 = page_height - elem["bbox_t"]
            y1 = page_height - elem["bbox_b"]

            if y0 > y1: # Đảm bảo y0 luôn nhỏ hơn y1
                y0, y1 = y1, y0

            rect = fitz.Rect(elem["bbox_l"], y0, elem["bbox_r"], y1)
            pix = page.get_pixmap(clip=rect)

            img_filename = f"image_page{elem['page']}_{i}.jpg"
            img_path = os.path.join(output_dir, img_filename)
            pix.save(img_path)

            elem["local_image_path"] = img_path

            # Upload lên Supabase nếu có client
            if supabase_client:
                remote_path = f"images/{doc_ai_id}/{img_filename}"
                try:
                    supabase_client.storage.from_("rag-data").upload(
                        path=remote_path,
                        file=img_path,
                        file_options={"content-type": "image/jpeg", "upsert": "true"}
                    )
                    public_url = supabase_client.storage.from_("rag-data").get_public_url(remote_path)
                    elem["image_ref"] = public_url
                except Exception as e:
                    print(f"     [Lỗi] Upload ảnh {img_filename} lên Supabase thất bại: {e}")
                    elem["image_ref"] = f"https://firebasestorage.googleapis.com/v0/b/your-app.appspot.com/o/{img_filename}?alt=media"
            else:
                elem["image_ref"] = f"https://firebasestorage.googleapis.com/v0/b/your-app.appspot.com/o/{img_filename}?alt=media"

            count += 1

    pdf.close()
    print(f" Đã cắt và lưu {count} hình ảnh vào thư mục '{output_dir}'.")
    return element_coords

# MODULE 4: Đóng Gói 5 Tủ JSON (Sẵn sàng gọi Embedding)
def package_chunks(element_coords, struct_merged=None, document_id="doc_001"):
    text_chunks = []
    image_chunks = []
    table_chunks = []
    formula_chunks = []
    code_chunks = []
    intro_chunks = [] # Tủ mới: Intro and Heading

    # 1. Quét qua struct_merged để tạo intro_chunks
    if struct_merged:
        for i, heading in enumerate(struct_merged):
            children = heading.get("heading_children", [])
            if children: # Chỉ tạo chunk nếu có heading con
                heading_text = heading.get("text", "Root")
                # Format câu văn theo ý tưởng của user
                children_str = '", "'.join(children)
                content = f'Mục "{heading_text}" chứa các nội dung "{children_str}"'

                chunk_id = f"{document_id}_intro_{i}"
                chunk = {
                    "chunk_id": chunk_id,
                    "document_id": document_id,
                    "page_number": heading.get("page", 1),
                    "heading_path": heading_text,
                    "content": content
                }
                intro_chunks.append(chunk)

    for i, elem in enumerate(element_coords):
        chunk_id = f"{document_id}_page{elem['page']}_chunk{i}"

        base_info = {
            "chunk_id": chunk_id,
            "document_id": document_id,
            "page_number": elem["page"],
            "heading_path": elem.get("heading_path", ""),
            "heading_parent": elem.get("heading_parent", ""),
            "heading_children": elem.get("heading_children", []),
        }

        if elem["type"] == "text":
            text_content = elem["text"]
            text_length = len(text_content)

            # Tính toán số lượng chunk dựa trên độ dài
            if text_length <= 800:
                num_chunks = 1
            elif 801 <= text_length <= 1000:
                num_chunks = 2
            elif 1001 <= text_length <= 1200:
                num_chunks = 3
            else:
                num_chunks = 4

            overlap = 200

            if num_chunks == 1:
                chunk = {**base_info, "content": text_content}
                text_chunks.append(chunk)
            else:
                # Thuật toán chia chunk với overlap cố định
                chunk_size = (text_length + overlap * (num_chunks - 1)) // num_chunks
                stride = chunk_size - overlap

                start = 0
                for c_idx in range(num_chunks):
                    end = min(start + chunk_size, text_length)

                    if c_idx == num_chunks - 1:
                        chunk_text = text_content[start:]
                    else:
                        chunk_text = text_content[start:end]

                    # Tạo chunk_id phân biệt cho các chunk nhỏ
                    sub_chunk_id = f"{chunk_id}_sub{c_idx+1}"
                    chunk = {**base_info, "chunk_id": sub_chunk_id, "content": chunk_text}
                    text_chunks.append(chunk)

                    start += stride

        elif elem["type"] == "picture":
            chunk = {
                **base_info,
                "image_ref": elem.get("image_ref", ""),
                "local_image_path": elem.get("local_image_path", ""),
                "caption": elem.get("caption", ""),
                "prev_text_snippet": elem.get("prev_text_snippet", ""),
                "next_text_snippet": elem.get("next_text_snippet", ""),
            }
            image_chunks.append(chunk)

        elif elem["type"] == "table":
            chunk = {
                **base_info,
                "caption": elem.get("caption", ""),
                "table_horizontal_text": elem.get("table_horizontal_text", ""), # Trường mới đọc theo chiều ngang
                "raw_structure": elem.get("text", ""),
                "prev_text_snippet": elem.get("prev_text_snippet", ""),
                "next_text_snippet": elem.get("next_text_snippet", ""),
            }
            table_chunks.append(chunk)

        elif elem["type"] == "formula":
            chunk = {
                **base_info,
                "latex": elem.get("text", ""), # Tạm dùng text, nếu có LaTeX thì thay đổi
                "prev_text_snippet": elem.get("prev_text_snippet", ""),
                "next_text_snippet": elem.get("next_text_snippet", ""),
            }
            formula_chunks.append(chunk)

        elif elem["type"] == "code":
            code_raw = elem.get("text", "")
            chunk = {
                **base_info,
                "code_raw": code_raw,
                "code_flattened": code_raw.replace("\n", " ").strip(),
                "shingles": code_raw.split("\n")[:5], # Cắt tạm 5 dòng đầu làm shingles
                "prev_text_snippet": elem.get("prev_text_snippet", ""),
                "next_text_snippet": elem.get("next_text_snippet", ""),
            }
            code_chunks.append(chunk)

    print(f" Đã đóng gói: {len(text_chunks)} Text, {len(image_chunks)} Image, {len(table_chunks)} Table, {len(formula_chunks)} Formula, {len(code_chunks)} Code, {len(intro_chunks)} Intro/Heading.")
    return {
        "text_chunks": text_chunks,
        "image_chunks": image_chunks,
        "table_chunks": table_chunks,
        "formula_chunks": formula_chunks,
        "code_chunks": code_chunks,
        "intro_chunks": intro_chunks
    }


ko dùng firebase storage nữa, dùng supabase

In [ ]:
!pip install supabase firebase-admin

tải file key trong firebase --- cái này với supabase nó dùng chung

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import time
import json
import gzip
import urllib.request
import os
import fitz
import firebase_admin
from firebase_admin import credentials, firestore
from supabase import create_client, Client
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

# 1. KẾT NỐI FIREBASE & SUPABASE
# Firebase (Chỉ dùng Firestore làm bưu điện gửi thư)
try:
    cred = credentials.Certificate("serviceAccountKey.json")
    firebase_admin.initialize_app(cred)
except ValueError:
    pass
db = firestore.client()

# Supabase (Kho chứa file khổng lồ thay thế Firebase Storage)
SUPABASE_URL = "https://yrmkrkcnmuqedqboarpe.supabase.co"
SUPABASE_KEY = "nhap_key_supabase_cua_ban_vao_day"
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

def upload_json_to_supabase(data_dict, file_name):
    # 1. Lưu tạm file JSON ra máy ảo Colab (Dùng Nén Gzip)
    local_path = f"/content/{file_name}.gz"
    with gzip.open(local_path, "wt", encoding="utf-8") as f:
        json.dump(data_dict, f, ensure_ascii=False)

    # 2. Bơm file nén lên Supabase
    supabase.storage.from_("rag-data").upload(
        path=f"{file_name}.gz",
        file=local_path,
        file_options={"content-type": "application/gzip", "upsert": "true"}
    )

    # 3. Lấy đường link public
    public_url = supabase.storage.from_("rag-data").get_public_url(f"{file_name}.gz")
    return public_url

# LISTENER: HÓNG VIỆC TỪ GIAO DIỆN UI
def on_ui_request_snapshot(col_snapshot, changes, read_time):
    for change in changes:
        if change.type.name in ['ADDED', 'MODIFIED']:
            doc_data = change.document.to_dict()
            doc_id = change.document.id

            if doc_data.get("status") == "pending":
                print(f"\n[DOCLING] Nhận yêu cầu xử lý PDF mới: {doc_data.get('document_id')}")
                db.collection("ui_to_docling_tasks").document(doc_id).update({
                    "status": "docling_processing",
                    "progress_msg": "Đang khởi động thuật toán..."
                })

                def update_progress(msg):
                    db.collection("ui_to_docling_tasks").document(doc_id).update({"progress_msg": msg})

                try:
                    pdf_url = doc_data.get("pdf_url")
                    doc_ai_id = doc_data.get("document_id")
                    local_pdf_path = f"temp_{doc_ai_id}.pdf"

                    print("  -> 1. Đang tải file PDF từ Cloud về Colab...")
                    update_progress("Bước 1/10: Tải file PDF từ Đám mây về Colab...")
                    urllib.request.urlretrieve(pdf_url, local_pdf_path)

                    print("  -> 2. Bắt đầu chạy bóc tách bằng Docling (Rất nặng, xin chờ)...")
                    update_progress("Bước 2/10: Docling đang đọc từng trang PDF (Mất vài phút)...")
                    pipeline_options = PdfPipelineOptions()
                    pipeline_options.do_ocr = False
                    pipeline_options.do_table_structure = True
                    pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
                    pipeline_options.generate_picture_images = True
                    pipeline_options.images_scale = 2.0

                    converter = DocumentConverter(
                        format_options={
                            InputFormat.PDF: PdfFormatOption(
                                pipeline_options=pipeline_options,
                                backend=PyPdfiumDocumentBackend
                            )
                        }
                    )
                    doc = converter.convert(local_pdf_path).document

                    print("  -> 3. Chạy hàm tổng hợp find_heading của bạn...")
                    update_progress("Bước 3/10: Trích xuất các Tiêu đề (Heading)...")
                    struct_merged, real_heading_muc_luc_level, heading_muc_luc_tree, element_coords, heading_con_sot_giua_2level, removed_headings_log = find_heading(local_pdf_path, doc)

                    print("  -> 4. Chém bỏ phần Mục Lục...")
                    update_progress("Bước 4/10: Đang loại bỏ Mục Lục rác...")
                    element_coords = filter_elements_after_toc(element_coords, struct_merged)

                    print("  -> 5. Gắn gia phả (Heading Path)...")
                    update_progress("Bước 5/10: Khâu nối dữ liệu vào Gia phả...")
                    element_coords = assign_heading_paths(element_coords, struct_merged)

                    print("  -> 6. Gắn ngữ cảnh sát rạt...")
                    update_progress("Bước 6/10: Liên kết ngữ cảnh thông minh...")
                    element_coords = assign_context_snippets(element_coords, max_chars=300)

                    print("  -> 7. Đọc nội dung Bảng theo chiều ngang...")
                    update_progress("Bước 7/10: Trích xuất dữ liệu Bảng...")
                    element_coords = extract_full_table_text(element_coords, doc)

                    print("  -> 8. Cắt ảnh ra từ PDF và tải lên Đám mây...")
                    update_progress("Bước 8/10: Đang cắt Ảnh và Upload (Có thể hơi lâu)...")
                    # Tải ảnh lên Supabase
                    element_coords = crop_and_upload_images(element_coords, local_pdf_path, output_dir=f"/content/cropped_images_{doc_ai_id}", supabase_client=supabase, doc_ai_id=doc_ai_id)

                    print("  -> 9. Đóng gói JSON chia vào 6 tủ (Bao gồm tủ Intro/Heading)...")
                    update_progress("Bước 9/10: Sắp xếp dữ liệu vào 6 Tủ...")
                    json_chunks = package_chunks(element_coords, struct_merged=struct_merged, document_id=doc_ai_id)

                    print("  -> 10. Bơm JSON tổng lên Supabase Storage...")
                    update_progress("Bước 10/10: Đóng gói Gzip và Nén lên Đám mây...")
                    supabase_url = upload_json_to_supabase(json_chunks, f"{doc_ai_id}_raw.json")
                    print(f"       (Đã bơm lên mây tại: {supabase_url})")

                    print("  -> 11. Gắn tag báo cho Tab Embedding vào việc!")
                    db.collection("docling_to_embedding_tasks").document(doc_ai_id).set({
                        "document_id": doc_ai_id,
                        "raw_json_url": supabase_url, # Gửi ĐƯỜNG LINK SUPABASE
                        "status": "pending",
                        "timestamp": firestore.SERVER_TIMESTAMP
                    })

                    db.collection("ui_to_docling_tasks").document(doc_id).update({"status": "docling_done"})
                    print(f" ĐÃ HOÀN TẤT BÓC TÁCH CHO TÀI LIỆU: {doc_ai_id} \n")

                    # Phát âm thanh báo hiệu xong
                    try:
                        from google.colab import output
                        output.eval_js('new Audio("https://actions.google.com/sounds/v1/alarms/beep_short.ogg").play()')
                    except ImportError:
                        pass

                    # DỌN DẸP BỘ NHỚ RAM COLAB
                    print("  -> Đang dọn dẹp bộ nhớ RAM...")
                    del doc
                    del converter
                    del element_coords
                    del struct_merged
                    del json_chunks
                    import gc
                    gc.collect()
                    print("  -> Đã giải phóng RAM, sẵn sàng cho file tiếp theo!")

                except Exception as e:
                    print(f"[LỖI DOCLING]: {e}")
                    db.collection("ui_to_docling_tasks").document(doc_id).update({"status": "error", "error": str(e)})

# CHẠY VÒNG LẶP VÔ HẠN
# Biến global để lưu trữ listener cũ (nếu có)
if 'docling_ui_watch' in globals():
    try:
        docling_ui_watch.unsubscribe()
        print(" Đã dọn dẹp listener cũ đang chạy ngầm.")
    except:
        pass

def start_docling_worker():
    global docling_ui_watch
    print(" Bắt đầu khởi động Tab Docling (Worker)...")
    ui_ref = db.collection("ui_to_docling_tasks")
    docling_ui_watch = ui_ref.on_snapshot(on_ui_request_snapshot)
    print(" Docling đang trực chiến hóng đơn hàng PDF! (Bấm nút Dừng ở Colab để thoát).")
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print(" Dừng Tab Docling.")
        docling_ui_watch.unsubscribe()

start_docling_worker()


TEST Docling

In [ ]:
# CELL TEST TỔNG HỢP: UPLOAD FILE + CHẠY DOCLING + TẢI JSON
import json
from google.colab import files
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

print(" BƯỚC 1: HÃY BẤM NÚT 'CHOOSE FILES' DƯỚI ĐÂY ĐỂ CHỌN 1 FILE PDF TỪ MÁY TÍNH:")
uploaded = files.upload()

if not uploaded:
    print(" Bạn chưa tải file nào lên hoặc đã bấm Hủy!")
else:
    # Tự động lấy đúng tên file PDF mà bạn vừa tải lên
    local_pdf_path = list(uploaded.keys())[0]
    test_id = "test_doc_" + local_pdf_path.replace(".pdf", "").replace(" ", "_")

    print(f"\n BƯỚC 2: Bắt đầu quét file: {local_pdf_path}")
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = False
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
    pipeline_options.generate_picture_images = True
    pipeline_options.images_scale = 2.0

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
                backend=PyPdfiumDocumentBackend
            )
        }
    )

    print("1. Docling đang đọc PDF (Sẽ mất vài phút tùy độ dài trang)...")
    doc = converter.convert(local_pdf_path).document

    print("2. Chạy bộ thuật toán Hậu xử lý (Custom Pipeline)...")
    # Đảm bảo bạn đã Run các Cell khai báo hàm này ở phía trên rồi nhé
    struct_merged, _, _, element_coords, _, _ = find_heading(local_pdf_path, doc)
    element_coords = filter_elements_after_toc(element_coords, struct_merged)
    element_coords = assign_heading_paths(element_coords, struct_merged)
    element_coords = assign_context_snippets(element_coords, max_chars=300)
    element_coords = extract_full_table_text(element_coords, doc)

    print("3. Đóng gói JSON chia vào 6 tủ...")
    json_chunks = package_chunks(element_coords, struct_merged=struct_merged, document_id=test_id)

    print("4. Đang lưu kết quả và tải về máy...")
    output_filename = f"ket_qua_{test_id}.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(json_chunks, f, ensure_ascii=False, indent=4)

    print(f" TEST THÀNH CÔNG! Trình duyệt sẽ tự pop-up tải file '{output_filename}' về ngay bây giờ!")
    files.download(output_filename)

code bị nhận nhầm thành text